# Субагент «Мышление» — автозапуск (ver3)

Запускайте ячейки **по порядку** (или `Runtime → Run all`):

| Ячейка | Что делает | Ожидаемый итог |
|---|---|---|
| | — | этот ноутбук самодостаточен: весь код внутри | — | |
| 2 (A) | зависимости, сборка llama-cpp, все модели Qwen2.5 (1.5B/3B/7B и те же три без цензуры) | `OK-A`, подсказки режимов CPU/T4 |
| 3 (C) | собирает API-сервер | `записан … строк: N` |
| 4 (D) | LLM + API + публичный туннель + сторож | `смоук: X.X с`, `THINKING_URL=`, `THINKING_TOKEN=` |
| 5 (E) | фон: снапшот событий + keep-alive 12 ч | `фон запущен: …` |
| 6 (F) | остановка (по необходимости) | `субагент остановлен` |
| 7 | **печатает адрес и готовую команду для ПК** | `АДРЕС ТУННЕЛЯ: https://…` |

После ячейки 4 (или 7 — там всё готово копированием) выполните на ПК:

```bash
python tools/thinking_cli.py set-url <THINKING_URL> <THINKING_TOKEN>
python tools/thinking_cli.py doctor
python tools/thinking_cli.py panel     # → http://127.0.0.1:8765
```

**Чтобы адрес не менялся изо дня в день**, задайте в ячейке D переменную
`CLOUDFLARE_TUNNEL_TOKEN` (Cloudflare Dashboard → Zero Trust → Tunnels →
Create tunnel → token): адрес станет постоянным, и `set-url` больше не нужен.

**Лимиты бесплатного Colab** (ориентиры, чётких квот нет): GPU-сессии обычно
2–4 часа, CPU-остаток бывает и ~28 часов — но это не гарантия. Если работа
встала — Colab: Сессия → Лимиты; лимиты часто возвращаются примерно через
сутки (или смените Google-аккаунт). Сначала работайте на GPU, потом переходите
на CPU (в панели это видно и подсказывается).

---

## ⚡ Как поднять (пошагово)

**Открывайте в своём обычном браузере** — в том, где вы уже залогинены в
Google. В браузере код-агента вход в Google-аккаунт блокируется («Couldn’t
sign you in: браузер не поддерживает JavaScript») — это защита Google от
автоматизированных браузеров. Пароль туда вводить не нужно и не нужно.

1. Откройте <https://colab.research.google.com> и загрузите
   `colab/thinking_agent_ver3.ipynb` (**Upload notebook**).
2. **Runtime → Run all** (ячейки A → C → D).
3. В **ячейке 2 (A)** убедитесь, что `PROFILE = "big"` — иначе возьмётся
   лёгкая 3B. Сильная модель `Qwen3-30B-A3B` весит 17 ГБ, качается минут 10–20.
4. В конце выполните **ячейку 7/7** — она напечатает готовый адрес туннеля
   и готовую команду для ПК. Искать адрес в выводе ячейки D больше не нужно.

**Как передать результат агенту**

Скопируйте из вывода ячейки 7/7 строку `АДРЕС ТУННЕЛЯ` и токен и вставьте
в чат — агент настроит ПК и прогонит полный тест субагента.

Либо вставьте вывод ячейки D (там строки `THINKING_URL=` и
`THINKING_TOKEN=`) — этого достаточно.


In [ ]:
#@title 1/7 · Исходники внутри ноутбука — загрузка не нужна
import os
os.makedirs('/content/tc', exist_ok=True)
print('ноутбук самодостаточен: каждая ячейка содержит свой код')

In [ ]:
%%writefile /content/tc/cell_a_setup.py
# === Субагент «Мышление», ячейка A (окружение + модель) ===
# Отличия от оригинала (только ускорение, страховка и выбор моделей):
#   * если модель уже лежит на диске — загрузка пропускается целиком;
#   * качаем НЕ всё, а выбранный профиль (THINKING_PROFILE): «light»,
#     «gpu», «strong» (14B), «big» (Qwen3-30B-A3B, 17 ГБ, MoE),
#     «coder» (32B для разработки), «uncensored», «all»;
#   * профиль задаётся переменной окружения — её ставит ячейка ноутбука
#     (одна строка PROFILE = "…" прямо перед запуском);
#   * при провале импорта CUDA-колеса чиним это одной командой с PyPI,
#     но только если в системе есть видеокарта (иначе не тратим время).
import glob
import importlib
import os
import subprocess
import sys
import time

START = time.time()
CUDA_INDEX = "https://abetlen.github.io/llama-cpp-python/whl/cu124/"
CPU_INDEX = "https://abetlen.github.io/llama-cpp-python/whl/cpu/"


def say(msg: str) -> None:
    print(msg, flush=True)


def pip(args, env=None, timeout=900):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                          env=env, capture_output=True, text=True, timeout=timeout)


def has_gpu() -> bool:
    if os.path.exists("/dev/nvidia0"):
        return True
    try:
        import torch                                  # noqa: PLC0415
        return bool(torch.cuda.is_available())
    except Exception:
        pass
    return False


def try_import() -> tuple[bool, str]:
    try:
        import llama_cpp
        importlib.reload(llama_cpp)
        from llama_cpp import server as _srv          # noqa: F401
        return True, ""
    except Exception as exc:
        return False, str(exc)[:200]


GPU = has_gpu()
say(f"python {sys.version.split()[0]}  GPU={GPU}")

# --- 1) зависимости ( idempotent: если стоят, pip отвечает за секунды) ---
r = pip(["fastapi", "uvicorn[standard]", "httpx", "huggingface_hub", "requests"])
say(f"deps rc={r.returncode}")

# --- 2) движок: готовое колесо, без компиляции ---
ok, why = try_import()
if not ok:
    if "libcudart" in why and GPU:            # есть GPU, но нет библиотек CUDA
        say("чиню CUDA-библиотеки с PyPI…")
        pip(["nvidia-cuda-runtime-cu12", "nvidia-cublas-cu12"])
        libdirs = []
        for mod in ("nvidia.cuda_runtime", "nvidia.cublas"):
            try:
                m = importlib.import_module(mod)
                libdirs.append(os.path.join(os.path.dirname(m.__file__), "lib"))
            except Exception:
                pass
        libdirs = [d for d in libdirs if os.path.isdir(d)]
        if libdirs:
            cur = os.environ.get("LD_LIBRARY_PATH", "")
            os.environ["LD_LIBRARY_PATH"] = ":".join(libdirs + ([cur] if cur else []))
            with open("/content/thinking_env.sh", "w", encoding="utf-8") as fh:
                fh.write("export LD_LIBRARY_PATH=" + os.environ["LD_LIBRARY_PATH"] + "\n")
            ok, why = try_import()

if not ok:
    # без GPU берём CPU-колесо: одна попытка, ~10 секунд вместо компиляции
    index = CUDA_INDEX if GPU else CPU_INDEX
    tag = "wheel-cu124" if GPU else "wheel-cpu"
    say(f"{tag}…")
    t0 = time.time()
    pip(["--no-cache-dir", "--extra-index-url", index, "llama-cpp-python[server]"])
    ok, why = try_import()
    say(f"  импорт: {'ОК' if ok else 'НЕ ОК — ' + why}  ({time.time()-t0:.0f}с)")

if not ok and GPU:                             # на CPU-wheel не вышло — пробуем CUDA
    say("fallback: wheel-cu124…")
    pip(["--no-cache-dir", "--extra-index-url", CUDA_INDEX, "llama-cpp-python[server]"])
    ok, why = try_import()
    if ok and "libcudart" in why:              # страховка от «битой» строки
        pass

assert ok, "не поставился llama-cpp-python: " + why
# Версию движка печатаем отдельно и всегда: от неё зависит, умеет ли llama.cpp
# архитектуру модели. Qwen3 появился в llama.cpp в 2025 году, и старая сборка
# на нём падает молча — в логе нет ни строки (прогон 04.10: так пропал
# Qwen3-30B-A3B, и пришлось гадать, память это или архитектура).
try:
    import llama_cpp
    say(f"движок: llama-cpp-python {getattr(llama_cpp, '__version__', 'версия неизвестна')}")
except Exception:
    say("движок: версию определить не удалось")

# --- 3) модель: сначала проверяем диск, потом качаем ---
MODEL_DIR = "/content/models/llm"
os.makedirs(MODEL_DIR, exist_ok=True)

have = [p for p in glob.glob(os.path.join(MODEL_DIR, "**", "*.gguf"), recursive=True)
        if os.path.getsize(p) > 600_000_000]

# Каталог: тег → (репозиторий, минимальный размер, подпись, ранг).
# Тег — точная подстрока имени файла: по нему узнаём свою модель и никогда
# не путаем соседние по имени (старые широкие теги «3b»/«uncensored» приняли
# uncensored-модель за обычную). Ранг — чем выше, тем сильнее модель.
from huggingface_hub import hf_hub_download, list_repo_files     # noqa: E402

CATALOG: dict[str, tuple[str, int, str, int]] = {
    "1.5b-instruct-q4": ("Qwen/Qwen2.5-1.5B-Instruct-GGUF", 600_000_000, "1.5B", 10),
    "3b-instruct-q4": ("bartowski/Qwen2.5-3B-Instruct-GGUF", 1_000_000_000, "3B", 20),
    "7b-instruct-q4": ("bartowski/Qwen2.5-7B-Instruct-GGUF", 3_000_000_000, "7B", 30),
    # Теги у больших моделей — строго те, что реально лежат в имени файла.
    # «14b-instruct-q4» не годится: он есть и в имени Coder-14B
    # (qwen2.5-coder-14b-instruct-q4_k_m.gguf) — и обычная 14B приняла бы
    # файл кодера. Поэтому «qwen2.5-14b-instruct» против «coder-14b».
    "qwen2.5-14b-instruct": ("bartowski/Qwen2.5-14B-Instruct-GGUF", 5_000_000_000,
                             "14B", 40),
    "coder-14b": ("bartowski/Qwen2.5-Coder-14B-Instruct-GGUF", 5_000_000_000,
                  "14B-CODE", 45),
    # MoE: 30B параметров, но активных только 3B — качество большой модели
    # при скорости небольшой. Единственный «сильный» вариант, который
    # осмысленно работает на бесплатном Colab (17.3 ГБ, RAM 12.6 ГБ — по
    # умолчанию mmap с диска, поэтому медленнее, но отвечает).
    "30b-a3b-q4": ("Qwen/Qwen3-30B-A3B-GGUF", 10_000_000_000, "30B-A3B", 60),
    "qwen2.5-32b-instruct": ("bartowski/Qwen2.5-32B-Instruct-GGUF", 10_000_000_000,
                             "32B", 70),
    "coder-32b": ("bartowski/Qwen2.5-Coder-32B-Instruct-GGUF", 10_000_000_000,
                  "32B-CODE", 80),
    "7b-instruct-uncensored": ("QuantFactory/Qwen2.5-7B-Instruct-Uncensored-GGUF",
                               3_000_000_000, "7B-UNC", 31),
    "3b-instruct-uncensored": ("mradermacher/Qwen2.5-3B-Instruct-Uncensored-GGUF",
                               1_000_000_000, "3B-UNC", 21),
    "1.5b-instruct-uncensored": ("mradermacher/Qwen2.5-1.5B-Instruct-uncensored-GGUF",
                                 600_000_000, "1.5B-UNC", 11),
}

# Профили: что качать. Первый элемент — активная модель профиля, остальные —
# запасные, если она не поднялась (ячейка D перебирает их по очереди).
PROFILES: dict[str, list[str]] = {
    "light": ["3b-instruct-q4", "1.5b-instruct-q4"],
    "gpu": ["7b-instruct-q4", "3b-instruct-q4", "1.5b-instruct-q4"],
    # Запасная модель здесь — 3B, а не кодерская 14B. Кодерская того же
    # размера, что и основная, поэтому в качестве запасной она бессмысленна:
    # если 14B не влезла в RAM, Coder-14B тоже не влезет — а качать её
    # пришлось бы зря (ещё 8.4 ГБ загрузки и диска). Запасная должна быть
    # заведомо меньше, тогда откат действительно спасает сессию.
    "strong": ["qwen2.5-14b-instruct", "3b-instruct-q4"],
    "big": ["30b-a3b-q4", "3b-instruct-q4"],
    "coder": ["coder-32b", "coder-14b", "3b-instruct-q4"],
    "uncensored": ["3b-instruct-uncensored", "3b-instruct-q4", "1.5b-instruct-q4"],
    "all": sorted(CATALOG, key=lambda t: -CATALOG[t][3]),
}
PROFILE = (os.environ.get("THINKING_PROFILE") or "auto").strip().lower()
if PROFILE in ("", "auto"):
    PROFILE = "gpu" if GPU else "light"
WANTED_TAGS = PROFILES.get(PROFILE) or PROFILES["light"]
say(f"профиль моделей: {PROFILE} → {', '.join(WANTED_TAGS)}")
if PROFILE not in PROFILES:
    say(f"  ⚠ неизвестный профиль, взят light (доступно: {', '.join(PROFILES)})")


def _has(tag: str) -> str:
    """Путь к уже скачанной модели по точному тегу (3b-instruct-q4, …)."""
    low = tag.lower()
    floor = CATALOG[tag][1]
    for p in have:
        name = os.path.basename(p).lower()
        if low in name and os.path.getsize(p) >= floor:
            return p
    return ""


def _fetch(repo: str, min_size: int) -> str:
    files = [f for f in list_repo_files(repo)
             if f.endswith(".gguf") and "Q4_K_M" in f.upper()
             and "IQ" not in f.upper()]
    if not files:
        say(f"  {repo}: подходящего .gguf нет")
        return ""
    t0 = time.time()
    got = hf_hub_download(repo_id=repo, filename=files[0], local_dir=MODEL_DIR)
    size = os.path.getsize(got)
    say(f"  {repo}: {size / 2**30:.2f} ГБ за {time.time() - t0:.0f}с")
    return got if size >= min_size else ""


MODELS: list[str] = []
tag_of: dict[str, str] = {}                  # путь → тег (для выбора активной)
for tag in WANTED_TAGS:
    repo, min_size, label, _rank = CATALOG[tag]
    path = _has(tag)
    if path:
        say(f"{tag}: уже на диске ({os.path.getsize(path) / 2**30:.2f} ГБ) — пропускаем")
        MODELS.append(path)
        tag_of[path] = tag
        continue
    say(f"{tag}: качаю {repo}…")
    try:
        path = _fetch(repo, min_size)
    except Exception as exc:                 # сеть отвалилась — не молчим
        say(f"  {repo}: {type(exc).__name__} {str(exc)[:100]}")
        path = ""
    if path:
        MODELS.append(path)
        tag_of[path] = tag
    else:
        say(f"  {tag}: не скачалась, пропускаем (работаем на остальных)")

if not MODELS:                               # аварийный план: что уже есть
    MODELS = have
    tag_of = {}
assert MODELS, "ни одна модель не нашлась"

# Активная — первая из профиля, что реально скачалась; если профиль не дал
# ничего, берём самую сильную из того, что лежит на диске.
active_tag = next((t for t in WANTED_TAGS if t in tag_of), "")
if active_tag:
    MODEL = tag_of[active_tag]
else:
    MODEL = max(MODELS, key=lambda p: os.path.getsize(p))
    say("  профиль пуст — взял самую большую модель с диска")


def _hint(p: str) -> str:
    """Подсказка режима (CPU/T4) — печатается при сборке и идёт в панель."""
    name = os.path.basename(p).lower()
    unc = " (без цензуры)" if "uncensored" in name else ""
    code = " · для кода" if "coder" in name else ""
    if "a3b" in name or "30b" in name:
        return (f"30B-A3B{code}{unc}: MoE — 30B параметров, активны 3B, "
                f"поэтому на CPU считает почти как 3B, а отвечает как большая")
    if "32b" in name:
        return (f"32B{code}{unc}: очень сильная, но на CPU медленная — "
                f"смени Runtime на T4 GPU")
    if "14b" in name:
        return (f"14B{code}{unc}: влезает в RAM целиком, на CPU ~2–4 ток/с")
    if "7b" in name:
        return (f"7B{unc}: смени Runtime → Change runtime type → T4 GPU "
                f"(на CPU ~1 ток/с)")
    if "3b" in name:
        return f"3B{unc}: работает на CPU (~2–3 ток/с), на T4 — быстрее"
    return f"1.5B{unc}: самая быстрая на CPU (~6 ток/с)"


say(f"ГОТОВО за {time.time()-START:.0f}с. Модели: "
    + ", ".join(f"{os.path.basename(p)} ({os.path.getsize(p) / 2**30:.2f} ГБ)"
                for p in MODELS))
for p in MODELS:
    say(f"  · {_hint(p)}")
if not GPU:
    say("  ⚠ Рантайм на CPU: большие модели качаются заранее, но считаются "
        "медленно — переключи Runtime на T4 GPU, если нужна скорость.")
say(f"активная: {MODEL}  ·  профиль: {PROFILE}  ·  "
    "переключение — в панели (вкладка «Модели»)")
with open("/content/thinking_model_path.txt", "w", encoding="utf-8") as fh:
    fh.write(MODEL)
with open("/content/thinking_models.txt", "w", encoding="utf-8") as fh:
    fh.write("\n".join(MODELS) + "\n")
with open("/content/thinking_backend.txt", "w", encoding="utf-8") as fh:
    fh.write("llama_cpp\n")
print("OK-A")

In [ ]:
%%writefile /content/thinking_server.py
# === Субагент «Мышление» — API (ячейка C) ===
# FastAPI + llama.cpp/vLLM. Субагент ТОЛЬКО советует: никаких инструментов,
# вызовов API изображений и записи файлов у него нет.
# Контракты полей должны совпадать с thinking/schemas.py (ловит tools/thinking_test.py).
from __future__ import annotations

import asyncio
import json
import os
import re
import secrets
import shutil
import subprocess
import sys
import time
import uuid
from collections import deque
from datetime import datetime, timezone
from typing import Any, Optional

import httpx
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.responses import HTMLResponse, StreamingResponse
from pydantic import BaseModel, Field, field_validator

# ---------------------------------------------------------------- конфиг ---
def _env_export(key: str, default: str = "") -> str:
    """Последнее значение `export KEY=...` из /content/thinking_env.sh.

    Туда ячейка D пишет THINKING_CTX — сервер и сторож должны брать одно и то же
    значение окна (AUD-08), и бюджет ответа тоже прижимается к нему.
    """
    try:
        for line in open("/content/thinking_env.sh", encoding="utf-8"):
            if line.startswith(f"export {key}="):
                default = line.split("=", 1)[1].strip().strip('"').strip("'")
    except OSError:
        pass
    return default


UPSTREAM = os.environ.get("THINKING_UPSTREAM",
                          "http://127.0.0.1:8001/v1/chat/completions")
MODEL_NAME = os.environ.get("THINKING_MODEL", "thinking")
TOKEN = os.environ.get("THINKING_TOKEN", "")
LLM_TIMEOUT = float(os.environ.get("THINKING_TIMEOUT", "300"))
MAX_TOKENS = int(os.environ.get("THINKING_MAX_TOKENS", "700"))


def _ctx_limit() -> int:
    """Сколько токенов ответа влезает в окно движка.

    Бюджеты в ячейке D задают длину ответа, а не «боязнь долгого ответа».
    Но окно модели (n_ctx) — жёсткий предел: промт плюс ответ должны в него
    уместиться, иначе llama.cpp обрезает промт или отказывает. Резервируем
    ~1200 токенов под системный промт, память и код активного файла.
    """
    try:
        gpu = os.path.exists("/dev/nvidia0") or bool(shutil.which("nvidia-smi"))
    except Exception:
        gpu = False
    ctx = int(_env_export("THINKING_CTX", "8192" if gpu else "4096") or 4096)
    return max(256, ctx - 1200)


MAX_TOKENS = min(MAX_TOKENS, _ctx_limit())
RPS = int(os.environ.get("THINKING_RPS", "5"))
CONCURRENCY = int(os.environ.get("THINKING_CONCURRENCY", "1"))
MAX_EVENTS, MAX_PLANS, PLAN_TTL_S = 1000, 64, 3600

ACTIONS = {"build", "test", "refactor", "verify", "docs", "prompt", "debug", "release"}
RETRY = {"none", "once", "exponential"}
SECRET_RE = re.compile(
    r"\b(password|passwd|secret|api[_-]?key|access[_-]?token|private[_-]?key|token)\b"
    r"\s*[=:]\s*\S+", re.I)


def utcnow() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def redact(text: str) -> str:
    return SECRET_RE.sub(lambda m: f"{m.group(1)}=[скрыто]", text or "")


# --------------------------------------------------------------- схемы ----
class PlanRequest(BaseModel):
    task: str = Field(..., min_length=3, max_length=2000)
    context: dict = Field(default_factory=dict)
    constraints: list[str] = Field(default_factory=list)
    max_steps: int = Field(12, ge=1, le=30)
    style_hint: Optional[str] = None


class PlanStep(BaseModel):
    id: int
    action: str = "verify"
    desc: str = Field(..., max_length=400)
    inputs: dict = Field(default_factory=dict)
    expected_output: Optional[str] = None
    retry_policy: str = "once"
    depends_on: list[int] = Field(default_factory=list)

    @field_validator("action", mode="before")
    @classmethod
    def _action(cls, v: Any) -> str:
        v = str(v or "").strip().lower()
        return v if v in ACTIONS else "verify"

    @field_validator("retry_policy", mode="before")
    @classmethod
    def _retry(cls, v: Any) -> str:
        v = str(v or "").strip().lower()
        return v if v in RETRY else "once"


class Plan(BaseModel):
    plan_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    source: str = "colab"
    rationale: str = ""
    confidence: float = 0.7
    goal: str
    sub_goals: list[str] = Field(default_factory=list)
    constraints: list[str] = Field(default_factory=list)
    contradictions: list[str] = Field(default_factory=list)
    steps: list[PlanStep]
    success_criteria: list[str] = Field(default_factory=list)
    unknown_files: list[str] = Field(default_factory=list)
    fallback: str = "выполнить напрямую, без субагента"
    created_at: str = Field(default_factory=utcnow)

    @field_validator("steps")
    @classmethod
    def _steps(cls, v: list[PlanStep]) -> list[PlanStep]:
        if not v:
            raise ValueError("план без шагов")
        if len(v) > 30:
            raise ValueError("слишком много шагов")
        ids = [s.id for s in v]
        if len(ids) != len(set(ids)):
            raise ValueError(f"дублирующиеся id шагов: {ids}")
        return v

    @field_validator("rationale", "goal", mode="before")
    @classmethod
    def _clean(cls, v: Any) -> Any:
        return redact(str(v or "")) if isinstance(v, str) else v


class ReflectRequest(BaseModel):
    plan_id: str
    step_id: int
    result: str = Field(..., max_length=4000)
    observation: Optional[str] = None
    error: Optional[str] = None


class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1, max_length=2000)
    context: dict = Field(default_factory=dict)
    history: list[dict] = Field(default_factory=list)
    memory: dict = Field(default_factory=dict)
    profile: Optional[str] = None
    max_steps: int = Field(4, ge=1, le=8)


class DevRequest(BaseModel):
    """Запрос режима «Разработка»: модель работает с кодом на ПК человека."""
    message: str = Field(..., min_length=1, max_length=3000)
    files: list[str] = Field(default_factory=list)
    active_name: str = ""
    active_code: str = ""


class ChatResponse(BaseModel):
    reply: str = ""
    rationale: str = ""
    steps: list[dict] = Field(default_factory=list)
    source: str = "colab"
    plan_id: str = ""
    fallback: bool = False
    memory_used: bool = False
    tokens_in: int = 0
    tokens_out: int = 0
    tokens_estimate: bool = True
    duration_ms: int = 0
    at: str = Field(default_factory=utcnow)

    @field_validator("reply", "rationale", mode="before")
    @classmethod
    def _clean(cls, v: Any) -> Any:
        return redact(str(v or "")) if isinstance(v, str) else v


class ReflectResponse(BaseModel):
    status: str = "ok"
    advice: str = ""
    rationale: str = ""
    next_steps: list[PlanStep] = Field(default_factory=list)
    updated_goal_stack: list[str] = Field(default_factory=list)


# ------------------------------------------------------------- состояние ---
class State:
    def __init__(self) -> None:
        self.events: deque[dict] = deque(maxlen=MAX_EVENTS)
        self.plans: dict[str, dict] = {}
        self.subs: set[asyncio.Queue] = set()
        self.seq = 0
        self.t0 = time.time()
        self.sem = asyncio.Semaphore(CONCURRENCY)
        self.win_t, self.win_n = time.time(), 0
        # переключение моделей на лету: процесс движка и флаг «идёт переключение»
        self.llm_proc: Optional[subprocess.Popen] = None
        self.switching = False
        self.stats = {"plans": 0, "reflects": 0, "chats": 0, "errors": 0,
                      "bad_json": 0, "llm_ms": deque(maxlen=200),
                      # честный учёт токенов: usage приходит не от каждого
                      # вызова, поэтому держим и «чистые» вызовы с usage
                      "tokens_in": 0, "tokens_out": 0, "usage_calls": 0}


S = State()   # единственный синглтон: снаружи глобальных переменных нет
app = FastAPI(title="Субагент «Мышление»", version="1.1")


@app.middleware("http")
async def cors(request: Request, call_next):
    resp = await call_next(request)
    resp.headers["Access-Control-Allow-Origin"] = "*"
    resp.headers["Access-Control-Allow-Headers"] = "Content-Type, X-Agent-Token"
    resp.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS"
    return resp


@app.options("/{path:path}")
async def options(path: str):
    return {"ok": True}


# -------------------------------------------------------------- события ---
async def emit(type_: str, text: str, **kw) -> dict:
    S.seq += 1
    ev = {"seq": S.seq, "type": type_, "text": redact(str(text)), "ts": utcnow(), **kw}
    S.events.append(ev)
    dead = []
    for q in list(S.subs):
        try:
            q.put_nowait(ev)
        except asyncio.QueueFull:
            # Подписчик отстал на 200 событий — рвём соединение: клиент
            # переподключится через /events?since= и доберёт хвост (AUD-11).
            # Раньше очередь была без лимита и медленный клиент копил
            # события в памяти сервера бесконечно.
            dead.append(q)
        except Exception:
            dead.append(q)
    for q in dead:
        S.subs.discard(q)
    return ev


# --------------------------------------------------- разбор JSON от LLM ---
def extract_json(text: str) -> dict:
    t = (text or "").strip()
    if not t:
        raise ValueError("пустой ответ LLM")
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", t).strip()
    try:
        out = json.loads(t)
        if isinstance(out, dict):
            return out
    except json.JSONDecodeError:
        pass
    start = t.find("{")
    if start < 0:
        raise ValueError("в ответе LLM нет '{'")
    depth, in_str, esc = 0, False, False
    for i in range(start, len(t)):
        ch = t[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                chunk = t[start:i + 1]
                try:
                    return json.loads(chunk)
                except json.JSONDecodeError:
                    return json.loads(re.sub(r",\s*([}\]])", r"\1", chunk))
    raise ValueError("несбалансированный JSON в ответе LLM")


# --------------------------------------------------------------- промты ---
SYSTEM_PLAN = """Ты — советник главного агента: опытный техлид, который помогает
распланировать задачу до начала работы. Тип проекта и технологию указывает
сам агент в поле КОНТЕКСТ — не привязывайся к одной технологии.
Ты только советуешь: инструментов, файлов и действий у тебя нет.
Отвечай ТОЛЬКО валидным JSON, без markdown-обёрток и текста вокруг.
Схема:
{"goal": str, "rationale": str, "confidence": 0.0..1.0,
 "sub_goals": [str], "constraints": [str], "contradictions": [str],
 "steps": [{"id": int, "action": str, "desc": str, "inputs": {},
            "expected_output": str, "retry_policy": "none|once|exponential",
            "depends_on": [int]}],
 "success_criteria": [str], "unknown_files": [str], "fallback": str}
Правила:
- goal, sub_goals, desc, contradictions, success_criteria, rationale — только по-русски.
- action — латинский тег ровно из: build test refactor verify docs prompt debug release.
- Упоминай только файлы из блока РАЗРЕШЁННЫЕ ФАЙЛЫ; если нужен другой — внеси его
  в unknown_files, но не выдумывай путь.
- Шаги проверяемые, по порядку, не больше указанного числа. Полей вне схемы не добавляй.
- rationale — 2-3 фразы по-русски: почему выбран именно такой план."""

SYSTEM_REFLECT = """Ты — рефлектор выполненного шага. Отвечай ТОЛЬКО JSON:
{"status": "ok|adjust|abort", "advice": str, "rationale": str,
 "next_steps": [{"id": int, "action": str, "desc": str, "inputs": {},
                 "expected_output": str, "retry_policy": str, "depends_on": [int]}],
 "updated_goal_stack": [str]}
advice и rationale — по-русски, коротко (до 240 символов). Ничего кроме JSON."""


SYSTEM_CHAT = """Ты — субагент «Мышление», собеседник человека-разработчика.
Отвечай обычным живым текстом по-русски — БЕЗ JSON, без markdown-обёрток
и кода-примеров вокруг ответа. Просто пиши, как обычный собеседник.
Правила:
- Сначала прямой ответ, потом (если уместно) 1-3 конкретных действия.
  Без воды и вступлений; до 700 символов, если вопрос не требует большего.
- Если в блоке ПАМЯТЬ есть факты о проекте — опирайся на них и не переспрашивай
  то, что уже известно.
- Если в блоке ДИАЛОГ есть прошлые реплики — не повторяй то, что уже обсудили.
- Ты советуешь: не выдумывай пути файлов, не обещай выполнить работу сам.
- Если вопрос про код и изменения файлов — скажи, что для этого есть вкладка
  «Разработка» (там модель меняет файлы подтверждением человека).
- Если не хватает данных — задай один-два уточняющих вопроса."""

CHAT_TURNS_MAX = 12
CHAT_HISTORY_CHARS = 2400


def build_chat_prompt(req: "ChatRequest") -> str:
    """Промт чата: память + последние реплики + сам вопрос."""
    facts = [str(f) for f in (req.memory.get("facts") or [])][-12:]
    mem_block = ""
    if req.profile:
        mem_block += f"О ПРОЕКТЕ: {str(req.profile)[:400]}\n"
    if facts:
        mem_block += "ПАМЯТЬ (факты, которые я уже сказал):\n"
        mem_block += "\n".join(f"- {f}" for f in facts)[:1200] + "\n"
    dialog = ""
    turns = [t for t in (req.history or []) if isinstance(t, dict)][-CHAT_TURNS_MAX:]
    if turns:
        lines = []
        for t in turns:
            who = "субагент" if t.get("role") == "subagent" else "я"
            lines.append(f"{who}: {str(t.get('text') or '')[:400]}")
        dialog = "ДИАЛОГ (недавно обсуждали):\n" + "\n".join(lines)[-CHAT_HISTORY_CHARS:] + "\n"
    ctx = json.dumps(req.context or {}, ensure_ascii=False)[:1500]
    return (mem_block + dialog +
            f"КОНТЕКСТ: {ctx}\n"
            f"ВОПРОС: {req.message[:2000]}\n"
            "Отвечай обычным текстом, без JSON.")


def build_plan_prompt(req: PlanRequest) -> str:
    ctx = json.dumps(req.context, ensure_ascii=False)
    files = req.context.get("files") or []
    tail = " | ".join(f"{e['type']}: {e['text'][:70]}" for e in list(S.events)[-10:])
    return (
        f"ЗАДАЧА: {req.task}\n"
        f"КОНТЕКСТ: {ctx[:4000]}\n"
        f"ОГРАНИЧЕНИЯ: {json.dumps(req.constraints, ensure_ascii=False)}\n"
        f"МАКС. ШАГОВ: {req.max_steps}\n"
        + (f"РАЗРЕШЁННЫЕ ФАЙЛЫ: {', '.join(str(f) for f in files)}\n" if files else "")
        + f"ПОСЛЕДНИЕ СОБЫТИЯ: {tail[:500]}\n"
        "Ответ — только JSON по схеме из system-промта."
    )


SYSTEM_DEV = """Ты — режим «Разработка» субагента «Мышление»: ты работаешь с кодом
человека, который физически лежит на его компьютере (ты сам файлы не трогаешь —
только присылаешь изменение, а применит его человек кнопкой).
Отвечай ТОЛЬКО валидным JSON, без markdown-обёрток и текста вокруг.
Схема:
{"action": "create|edit|none", "filename": str, "code": str, "comment": str}
Правила:
- action="create" — новый файл в рабочей папке; action="edit" — замена
  СУЩЕСТВУЮЩЕГО файла из списка ФАЙЛЫ (имя возьми ровно оттуда);
  action="none" — вопрос или обсуждение без изменения кода (code пустой).
- filename — только имя файла (без каталогов), обычно с расширением .py.
- code — ПОЛНОЕ новое содержимое файла целиком (не диф, не фрагмент):
  иначе применение частями непонятно. Только код, без markdown и пояснений
  внутри кода.
- comment — по-русски, 1-3 предложения: что изменено и что это даёт.
- Пиши рабочий Python: без TODO, без заглушек, с обработкой ошибок там,
  где они нужны; стандартная библиотека (у человека только она).
- Если данных мало или просят невозможное — action="none" и задай вопрос
  в comment. Ничего кроме JSON."""


def build_dev_prompt(req: "DevRequest") -> str:
    """Промт режима разработчика: файлы + открытый файл + запрос человека."""
    files = ", ".join(str(f) for f in (req.files or [])[:80]) or "—"
    out = (f"ФАЙЛЫ В РАБОЧЕЙ ПАПКЕ: {files}\n"
           f"ЗАПРОС: {req.message[:3000]}\n")
    if req.active_name:
        out += (f"ОТКРЫТ СЕЙЧАС ФАЙЛ: {req.active_name}\n"
                f"ЕГО КОД:\n{req.active_code[:8000]}\n")
    return out + "Ответ — только JSON по схеме из system-промта."


async def dev_data(raw: str) -> dict:
    """Разбор ответа режима разработчика: JSON либо честный текст."""
    try:
        data = extract_json(raw)
        action = str(data.get("action") or "none").lower()
        if action not in ("create", "edit", "none"):
            action = "none"
        return {"action": action,
                "filename": str(data.get("filename") or "")[:200],
                "code": str(data.get("code") or "")[:20000],
                "comment": redact(str(data.get("comment") or ""))[:1200]}
    except Exception:
        S.stats["bad_json"] += 1
        text = _plain_reply(raw)
        # Модель ответила текстом — пусть идёт комментарием, файлы не трогаем
        return {"action": "none", "filename": "", "code": "",
                "comment": redact(text)[:1200]}


# ------------------------------------------------------- вызовы LLM -------
async def chat_json(system: str, user: str, max_tokens: int = MAX_TOKENS) -> str:
    text, _ = await chat_json_usage(system, user, max_tokens)
    return text


async def chat_json_usage(system: str, user: str,
                          max_tokens: int = MAX_TOKENS) -> tuple[str, dict]:
    """Как chat_json, но возвращает ещё и usage — по нему считаются токены.

    llama.cpp-сервер отдаёт usage не всегда, поэтому при отсутствии честно
    отдаём пустой dict: ПК посчитает оценку по длине текста сам.
    """
    async with S.sem:
        t = time.time()
        async with httpx.AsyncClient(timeout=LLM_TIMEOUT) as c:
            r = await c.post(UPSTREAM, json={
                "model": MODEL_NAME, "stream": False, "temperature": 0.2,
                "max_tokens": max_tokens,
                "messages": [{"role": "system", "content": system},
                             {"role": "user", "content": user}],
            })
            r.raise_for_status()
            data = r.json()
        S.stats["llm_ms"].append(int((time.time() - t) * 1000))
    usage = data.get("usage") or {}
    if isinstance(usage, dict) and usage:
        S.stats["tokens_in"] += int(usage.get("prompt_tokens") or 0)
        S.stats["tokens_out"] += int(usage.get("completion_tokens") or 0)
        S.stats["usage_calls"] += 1
    return data["choices"][0]["message"]["content"], (usage if isinstance(usage, dict) else {})


async def with_heartbeat(src, gap: float = 15.0):
    """Отдаёт SSE-комментарии, пока модель молчит.

    Cloudflare рвёт соединение по 524, если не получил первые байты ТЕЛА
    ответа дольше ~120 с. Заголовки приходят мгновенно, а тело — только
    когда модель выдаст первый токен, и на 14B на CPU это больше минуты
    (04.10: `ask` и обычный `/chat` оба падали в 524, хотя модель жива).
    Поэтому шлём `: ping` сразу и далее каждые `gap` секунд: это валидный
    SSE-комментарий, который прокси пропускает, а клиент игнорирует.

    Наполнение идёт отдельной задачей: иначе таймаут отменял бы чтение
    модели на полуслове и поток вёлся бы в никуда.
    """
    q: asyncio.Queue = asyncio.Queue()
    done = object()

    async def _pump() -> None:
        try:
            async for item in src:
                await q.put(item)
        except asyncio.CancelledError:
            raise
        except Exception as exc:
            await q.put(exc)
        await q.put(done)

    task = asyncio.create_task(_pump())
    try:
        yield ": open\n\n"                     # первый байт тела — сразу
        while True:
            try:
                item = await asyncio.wait_for(q.get(), timeout=gap)
            except asyncio.TimeoutError:
                yield f": ping {int(time.time())}\n\n"
                continue
            if item is done:
                return
            if isinstance(item, BaseException):
                raise item
            yield item
    finally:
        task.cancel()


async def chat_stream(system: str, user: str, max_tokens: int = MAX_TOKENS):
    async with httpx.AsyncClient(timeout=LLM_TIMEOUT) as c:
        async with c.stream("POST", UPSTREAM, json={
            "model": MODEL_NAME, "stream": True, "temperature": 0.2,
            "max_tokens": max_tokens,
            "messages": [{"role": "system", "content": system},
                         {"role": "user", "content": user}],
        }) as r:
            r.raise_for_status()
            async for line in r.aiter_lines():
                if not line.startswith("data:"):
                    continue
                chunk = line[5:].strip()
                if chunk == "[DONE]":
                    break
                try:
                    d = json.loads(chunk)
                except Exception:
                    continue
                piece = d.get("choices", [{}])[0].get("delta", {}).get("content")
                if piece:
                    yield piece


# ------------------------------------------------- токен + rate limit -----
def _auth(x_agent_token: str, query_token: str) -> None:
    got = x_agent_token or query_token
    if TOKEN and got != TOKEN:
        raise HTTPException(401, "неверный токен")
    now = time.time()
    if now - S.win_t >= 1.0:
        S.win_t, S.win_n = now, 0
    S.win_n += 1
    if S.win_n > RPS:
        raise HTTPException(429, "слишком часто (rate limit)")


async def _guard(raw: str) -> dict:
    try:
        return extract_json(raw)
    except Exception as exc:
        S.stats["bad_json"] += 1
        try:
            with open("/content/last_bad_json.txt", "w", encoding="utf-8") as fh:
                fh.write(raw or "")
        except Exception:
            pass
        await emit("error", f"LLM вернула не-JSON: {exc}")
        raise HTTPException(502, f"LLM вернула не-JSON: {exc}")


def _plain_reply(raw: str) -> str:
    """Ответ без JSON — просто текст.

    Маленькие модели (1.5B) часто отвечают обычной фразой вместо JSON.
    Раньше такой ответ считался поломкой (HTTP 502 «пустой ответ»), хотя
    человек-то получил осмысленный текст. Теперь текст идёт как есть,
    а счётчик plain_text показывает, что модель ушла от формата.
    """
    t = (raw or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", t).strip()
    return t


async def guard_reply(raw: str) -> dict:
    """Как _guard, но не падает на обычном тексте: отдаёт {'reply': текст}."""
    try:
        data = extract_json(raw)
        if str(data.get("reply") or "").strip():
            return data
    except Exception as exc:
        S.stats["bad_json"] += 1
        await emit("error", f"LLM ответила текстом, а не JSON: {exc}")
    text = _plain_reply(raw)
    if not text:
        raise HTTPException(502, "пустой ответ чата")
    # Частая форма: модель обернула JSON-объект в текст, но не довела его до
    # конца (обрезали по max_tokens). Тогда разбираем поле reply вручную.
    m = re.search(r'"reply"\s*:\s*"(.*?)"\s*(?:,|})', text, re.S)
    if m:
        text = m.group(1).replace('\\"', '"').replace("\\n", "\n").strip()
    S.stats["plain_text"] = int(S.stats.get("plain_text", 0)) + 1
    return {"reply": text, "rationale": "", "steps": []}


PLAN_LIST_FIELDS = ("sub_goals", "constraints", "contradictions",
                    "success_criteria", "unknown_files")


def coerce_plan(data: dict) -> dict:
    """Мягко приводит план к схеме.

    Маленькие модели часто ошибаются в типах: список пишут строкой, а id шага
    — словом вроде "s1". Раньше это was 500; теперь такое просто чинится.
    """
    out = dict(data or {})
    for key in PLAN_LIST_FIELDS:
        v = out.get(key)
        if v is None:
            continue
        if isinstance(v, str):
            out[key] = [v] if v.strip() else []
        elif not isinstance(v, list):
            out[key] = [str(v)]
    steps = out.get("steps")
    if isinstance(steps, list):
        norm = []
        for i, s in enumerate(steps, 1):
            if isinstance(s, dict):
                try:
                    sid = int(s.get("id"))
                except (TypeError, ValueError):
                    sid = i
                norm.append({**s, "id": sid})
            else:
                norm.append({"id": i, "desc": str(s)[:300]})
        out["steps"] = norm
    return out


async def plan_data(raw: str, task: str) -> dict:
    """Как _guard, но обычный текст модели превращается в план из одного шага.

    Иначе слабая модель (1.5B) давала бы ошибку вместо полезного ответа,
    хотя рассуждать она может — просто не в том формате.
    """
    try:
        data = extract_json(raw)
        if isinstance(data.get("steps"), list) and data["steps"]:
            return coerce_plan(data)
    except Exception as exc:
        S.stats["bad_json"] += 1
        await emit("error", f"LLM ответила текстом, а не JSON: {exc}")
    text = _plain_reply(raw)
    if not text:
        raise HTTPException(502, "пустой ответ плана")
    S.stats["plain_text"] = int(S.stats.get("plain_text", 0)) + 1
    return coerce_plan({
        "goal": str(task or "")[:300] or "(задача не названа)",
        "rationale": text[:600],
        # id у шага — число: серверная схема жёстко требует int,
        # строковый "s1" приводил к 500 на валидации.
        "steps": [{"id": 1, "desc": text[:300]}],
        "confidence": 0.3,
    })


async def reflect_data(raw: str) -> dict:
    """Как _guard, но текстовая рефлексия тоже считается ответом."""
    try:
        return extract_json(raw)
    except Exception as exc:
        S.stats["bad_json"] += 1
        await emit("error", f"LLM ответила текстом, а не JSON: {exc}")
    text = _plain_reply(raw)
    if not text:
        raise HTTPException(502, "пустой ответ разбора")
    S.stats["plain_text"] = int(S.stats.get("plain_text", 0)) + 1
    return {"status": "ok", "advice": text[:800], "rationale": "", "next_steps": []}


def _store(plan: Plan) -> None:
    S.plans[plan.plan_id] = plan.model_dump()
    if len(S.plans) > MAX_PLANS:
        for key in sorted(S.plans, key=lambda k: S.plans[k].get("created_at", ""))[:8]:
            S.plans.pop(key, None)


# ---------------------------------------------------------- эндпоинты -----
@app.get("/", response_class=HTMLResponse)
async def index(token: str = ""):
    return PANEL_HTML.replace("__TOKEN__", token)


@app.get("/health")
async def health(x_agent_token: str = Header(default=""), token: str = ""):
    _auth(x_agent_token, token)
    gpu, vram = "cpu", None
    try:
        import torch
        gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
        vram = round(torch.cuda.memory_allocated() / 2 ** 30, 2)
    except Exception:
        pass
    return {"status": "ok", "model": MODEL_NAME, "upstream": UPSTREAM,
            "uptime_s": int(time.time() - S.t0), "plans": len(S.plans),
            "events": S.seq, "gpu": gpu, "vram_used_gb": vram}


@app.get("/metrics")
async def metrics(x_agent_token: str = Header(default=""), token: str = ""):
    _auth(x_agent_token, token)
    ms = list(S.stats["llm_ms"])
    calls = S.stats["plans"] + S.stats["reflects"] + S.stats["chats"]
    return {"events": S.seq, "plans_total": S.stats["plans"],
            "reflects": S.stats["reflects"], "chats_total": S.stats["chats"],
            "errors": S.stats["errors"],
            "bad_json": S.stats["bad_json"], "subscribers": len(S.subs),
            "llm_ms_avg": int(sum(ms) / len(ms)) if ms else None,
            "llm_ms_p95": sorted(ms)[int(len(ms) * 0.95)] if len(ms) > 5 else None,
            "queue": CONCURRENCY,
            # usage отдаёт не каждый вызов — панель помечает такие цифры оценкой
            "tokens_in": S.stats["tokens_in"], "tokens_out": S.stats["tokens_out"],
            "tokens_total": S.stats["tokens_in"] + S.stats["tokens_out"],
            "usage_calls": S.stats["usage_calls"],
            "usage_exact": bool(calls) and S.stats["usage_calls"] >= calls}


@app.get("/events")
async def events(since: int = 0, tail: int = 0,
                 x_agent_token: str = Header(default=""), token: str = ""):
    _auth(x_agent_token, token)
    items = list(S.events)
    if since:
        items = [e for e in items if e["seq"] > since]
    if tail:
        items = items[-tail:]
    return {"last_seq": S.seq, "events": items}


@app.get("/events/stream")
async def events_stream(x_agent_token: str = Header(default=""), token: str = ""):
    _auth(x_agent_token, token)

    async def gen():
        for e in list(S.events)[-50:]:
            yield f"data: {json.dumps(e, ensure_ascii=False)}\n\n"
        # Очередь с лимитом: медленный подписчик не должен копить события
        # в памяти сервера (AUD-11) — при переполнении его отключит emit()
        q: asyncio.Queue = asyncio.Queue(maxsize=200)
        S.subs.add(q)
        try:
            while True:
                try:
                    # ping каждые 10 с: туннель и прокси рвут «тихое» соединение,
                    # а панель на ПК держит read-timeout потока 40 с
                    ev = await asyncio.wait_for(q.get(), timeout=10)
                    yield f"data: {json.dumps(ev, ensure_ascii=False)}\n\n"
                except asyncio.TimeoutError:
                    yield ": ping\n\n"
        finally:
            S.subs.discard(q)

    return StreamingResponse(gen(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache",
                                      "X-Accel-Buffering": "no"})


@app.post("/plan")
async def plan(req: PlanRequest, x_agent_token: str = Header(default=""), token: str = ""):
    _auth(x_agent_token, token)
    await emit("thought", f"Задача: {req.task[:150]}")
    await emit("thought", "Декомпозирую на шаги…")
    try:
        raw = await chat_json(SYSTEM_PLAN, build_plan_prompt(req))
        data = await plan_data(raw, req.task)
        p = Plan(**data)
        _store(p)
        S.stats["plans"] += 1
        await emit("rationale", p.rationale or "(без объяснения)", plan_id=p.plan_id)
        for c in p.contradictions:
            await emit("contradiction", c, plan_id=p.plan_id)
        for s in p.steps:
            await emit("plan_step", f"[{s.id}] {s.desc}", plan_id=p.plan_id, step_id=s.id)
        await emit("final", f"План {p.plan_id[:8]} готов: {len(p.steps)} шагов",
                   plan_id=p.plan_id)
        return p
    except HTTPException:
        raise
    except httpx.HTTPError as exc:
        S.stats["errors"] += 1
        await emit("error", f"LLM backend недоступен: {type(exc).__name__}")
        raise HTTPException(503, f"upstream недоступен: {type(exc).__name__}")
    except Exception as exc:
        S.stats["errors"] += 1
        await emit("error", str(exc)[:300])
        raise HTTPException(500, str(exc)[:300])


@app.post("/plan/stream")
async def plan_stream(req: PlanRequest, x_agent_token: str = Header(default=""), token: str = ""):
    _auth(x_agent_token, token)
    await emit("thought", f"Стрим плана: {req.task[:150]}")

    async def gen():
        buf: list[str] = []
        try:
            async with S.sem:
                async for piece in chat_stream(SYSTEM_PLAN, build_plan_prompt(req)):
                    buf.append(piece)
                    yield f"data: {json.dumps({'type': 'token', 'text': piece}, ensure_ascii=False)}\n\n"
            data = await plan_data("".join(buf), req.task)
            p = Plan(**data)
            _store(p)
            S.stats["plans"] += 1
            yield f"data: {json.dumps({'type': 'rationale', 'text': p.rationale}, ensure_ascii=False)}\n\n"
            for s in p.steps:
                yield ("data: " + json.dumps({"type": "plan_step",
                                              "text": f"[{s.id}] {s.desc}",
                                              "step_id": s.id},
                                             ensure_ascii=False) + "\n\n")
            final = json.dumps(p.model_dump(), ensure_ascii=False, default=str)
            yield ("data: " + json.dumps({"type": "final", "text": final,
                                          "plan_id": p.plan_id},
                                         ensure_ascii=False) + "\n\n")
        except Exception as exc:
            S.stats["errors"] += 1
            yield ("data: " + json.dumps({"type": "error", "text": str(exc)[:300]},
                                         ensure_ascii=False) + "\n\n")

    return StreamingResponse(with_heartbeat(gen()), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache",
                                      "X-Accel-Buffering": "no"})


@app.post("/reflect")
async def reflect(req: ReflectRequest, x_agent_token: str = Header(default=""), token: str = ""):
    _auth(x_agent_token, token)
    stored = S.plans.get(req.plan_id)
    plan_json = json.dumps(stored, ensure_ascii=False, default=str) if stored \
        else "план не найден (возможно, после рестарта сервера)"
    await emit("thought", f"Рефлексия шага {req.step_id}",
               plan_id=req.plan_id, step_id=req.step_id)
    try:
        raw = await chat_json(SYSTEM_REFLECT,
                              f"ПЛАН: {plan_json[:3000]}\nШАГ: {req.step_id}\n"
                              f"РЕЗУЛЬТАТ: {req.result[:2000]}\n"
                              f"НАБЛЮДЕНИЕ: {req.observation or '-'}\n"
                              f"ОШИБКА: {req.error or '-'}",
                              max_tokens=min(500, _ctx_limit()))
        data = await reflect_data(raw)
        status = data.get("status")
        if status not in ("ok", "adjust", "abort"):
            status = "ok"
        steps = []
        for item in (data.get("next_steps") or [])[:8]:
            try:
                steps.append(PlanStep(**item))
            except Exception:
                continue
        out = ReflectResponse(
            status=status,
            advice=redact(str(data.get("advice") or ""))[:800],
            rationale=redact(str(data.get("rationale") or ""))[:400],
            next_steps=steps,
            updated_goal_stack=[str(x) for x in (data.get("updated_goal_stack") or [])][:12],
        )
        S.stats["reflects"] += 1
        await emit("final" if out.status == "ok" else "contradiction",
                   f"[{out.status}] {out.advice}", plan_id=req.plan_id, step_id=req.step_id)
        return out
    except HTTPException:
        raise
    except Exception as exc:
        S.stats["errors"] += 1
        await emit("error", f"рефлексия: {str(exc)[:200]}")
        raise HTTPException(500, str(exc)[:300])


@app.post("/cancel/{plan_id}")
async def cancel(plan_id: str, x_agent_token: str = Header(default=""), token: str = ""):
    _auth(x_agent_token, token)
    S.plans.pop(plan_id, None)
    await emit("thought", f"План {plan_id[:8]} отменён")
    return {"status": "cancelled", "plan_id": plan_id}


@app.post("/chat")
async def chat(req: ChatRequest, x_agent_token: str = Header(default=""), token: str = ""):
    """Живой диалог человека с субагентом: память + последние реплики + ответ."""
    _auth(x_agent_token, token)
    t0 = time.time()
    memory = req.memory if isinstance(req.memory, dict) else {}
    facts = [str(f) for f in (memory.get("facts") or []) if str(f).strip()]
    used = bool(facts or req.profile or req.history)
    await emit("thought", f"Чат: {req.message[:120]}")
    try:
        raw, usage = await chat_json_usage(SYSTEM_CHAT, build_chat_prompt(req), 500)
        data = await guard_reply(raw)
        reply = str(data.get("reply") or "").strip()
        if not reply:                      # модель вернула пустоту — честно об этом
            raise HTTPException(502, "пустой ответ чата")
        steps = []
        for item in (data.get("steps") or [])[:5]:
            desc = item.get("desc") if isinstance(item, dict) else item
            if str(desc or "").strip():
                steps.append({"desc": str(desc)[:300]})
        S.stats["chats"] += 1
        tin = int(usage.get("prompt_tokens") or 0)
        tout = int(usage.get("completion_tokens") or 0)
        out = ChatResponse(reply=reply, rationale=str(data.get("rationale") or "")[:600],
                           steps=steps, source="colab", memory_used=used,
                           tokens_in=tin, tokens_out=tout,
                           tokens_estimate=not bool(usage),
                           duration_ms=int((time.time() - t0) * 1000))
        await emit("final", f"[чат] {reply[:200]}")
        if tin or tout:
            await emit("thought", f"токены: в {tin}, out {tout}")
        return out
    except HTTPException:
        raise
    except httpx.HTTPError as exc:
        S.stats["errors"] += 1
        await emit("error", f"чат: upstream недоступен: {type(exc).__name__}")
        raise HTTPException(503, "LLM backend недоступен")
    except Exception as exc:
        S.stats["errors"] += 1
        await emit("error", f"чат: {str(exc)[:200]}")
        raise HTTPException(500, str(exc)[:300])


# ------------------------------------------- выгрузка логов (для архива) ----
DUMP_FILES = {
    "llm.log": "/content/llm.log",
    "api.log": "/content/thinking_api.log",
    "tunnel.log": "/content/tunnel.log",
    "server.py": "/content/thinking_server.py",
    "snapshot.json": "/content/thinking_snapshot.json",
    "last_bad_json.txt": "/content/last_bad_json.txt",
    "model_path.txt": "/content/thinking_model_path.txt",
}


# ------------------------------------------------- переключение моделей ---
MODELS_FILE = "/content/thinking_models.txt"
ACTIVE_MODEL_FILE = "/content/thinking_active_model.txt"
LLM_LOG = "/content/llm.log"
LLM_HOST, LLM_PORT = "127.0.0.1", 8001
S.llm_proc: Optional[subprocess.Popen] = None   # noqa: E305


def _model_list() -> list[str]:
    """Модели, доступные рантайму: то, что перечислила ячейка A/D."""
    try:
        with open(MODELS_FILE, encoding="utf-8") as fh:
            items = [ln.strip() for ln in fh if ln.strip()]
    except OSError:
        items = []
    return [p for p in items if os.path.exists(p)]


def _active_model() -> str:
    try:
        with open(ACTIVE_MODEL_FILE, encoding="utf-8") as fh:
            cur = fh.read().strip()
            if cur and os.path.exists(cur):
                return cur
    except OSError:
        pass
    # Не угадываем по списку: иначе панель решит, что активна первая модель,
    # а в рантайме может крутиться другая — и переключение не сработает.
    return ""


def _model_label(path: str) -> str:
    """Короткое имя для интерфейса: qwen2.5-3b-instruct-q4_k_m → 3B.

    Разные сборки различаются суффиксом (UNC — uncensored), иначе в панели
    две 7B выглядят одинаково и переключение выбирает не ту.
    """
    name = os.path.basename(path).lower()
    size = ""
    for tag in ("0.5b", "1.5b", "3b", "7b", "14b"):
        if tag in name:
            size = tag.upper()
            break
    if size:
        if "uncensored" in name or "unfiltered" in name:
            return size + "-UNC"
        if "dolphin" in name:
            return size + "-DOLPHIN"
        return size
    return os.path.splitext(os.path.basename(path))[0][:24]


def _model_hint(path: str, gpu: Optional[bool] = None) -> str:
    """Подсказка нужного режима (CPU / T4 GPU) — видна в панели и CLI.

    Пользователь меняет hardware accelerator в Colab, поэтому подсказка
    учитывает и модель, и то, что запущено прямо сейчас.
    """
    if gpu is None:
        gpu = os.path.exists("/dev/nvidia0") or bool(shutil.which("nvidia-smi"))
    name = os.path.basename(path).lower()
    here = "сейчас T4/GPU" if gpu else "сейчас CPU"
    unc = " Сборка без цензуры." if "uncensored" in name else ""
    if any(tag in name for tag in ("7b", "8b", "14b")):
        if gpu:
            return (f"7B · {here} — считается быстро (~40–60 ток/с).{unc}")
        return (f"7B · {here} — на CPU это ~1 ток/с, очень медленно. "
                f"Для лучшей работы смени Runtime → Change runtime type → "
                f"T4 GPU.{unc}")
    if "3b" in name:
        return (f"3B · {here} — на CPU ~2–3 ток/с, на T4 ~40 ток/с; "
                f"компромисс скорости и качества.{unc}")
    if "1.5b" in name or "0.5b" in name:
        return (f"1.5B · {here} — самая быстрая на CPU (~6 ток/с), "
                f"лучший выбор без видеокарты.{unc}")
    return f"{here}; на T4 любая модель считается заметно быстрее.{unc}"


def _llm_cmd(model_path: str) -> list[str]:
    """Та же строка запуска, что и в ячейке D, но с нужной моделью."""
    gpu = os.path.exists("/dev/nvidia0") or bool(shutil.which("nvidia-smi"))
    # n_ctx — из thinking_env.sh (пишет ячейка D), чтобы запуск и сторожевый
    # перезапуск не разошлись (AUD-08); фолбэк — те же значения, что в D
    ctx = _env_export("THINKING_CTX", "8192" if gpu else "4096")
    return [sys.executable, "-m", "llama_cpp.server",
            "--model", model_path,
            "--n_ctx", ctx,
            "--n_gpu_layers", "-1",
            "--host", LLM_HOST, "--port", str(LLM_PORT),
            "--model_alias", MODEL_NAME,
            "--n_threads", "4", "--n_batch", "512", "--verbose", "False"]


async def _llm_ready(tries: int = 90, gap: float = 2.0) -> bool:
    url = f"http://{LLM_HOST}:{LLM_PORT}/v1/models"
    for _ in range(tries):
        try:
            async with httpx.AsyncClient(timeout=5) as c:
                r = await c.get(url)
            if r.status_code == 200:
                return True
        except httpx.HTTPError:
            pass
        await asyncio.sleep(gap)
    return False


async def _switch_to(path: str) -> dict:
    """Убивает текущий движок и поднимает на выбранной модели."""
    S.switching = True
    try:
        await emit("thought", f"Переключаю модель на {_model_label(path)}…")
        subprocess.run(["pkill", "-f", "llama_cpp.server"], capture_output=True)
        await asyncio.sleep(2)
        log = open(LLM_LOG, "a", encoding="utf-8")
        S.llm_proc = subprocess.Popen(_llm_cmd(path), stdout=log, stderr=subprocess.STDOUT)
        if not await _llm_ready():
            raise RuntimeError(f"{_model_label(path)} не поднялась")
        try:
            with open(ACTIVE_MODEL_FILE, "w", encoding="utf-8") as fh:
                fh.write(path)
        except OSError:
            pass
        S.stats["model_switches"] = int(S.stats.get("model_switches", 0)) + 1
        await emit("final", f"Модель переключена: {_model_label(path)}")
        return {"ok": True, "active": path, "label": _model_label(path)}
    finally:
        S.switching = False


@app.get("/models")
async def models(x_agent_token: str = Header(default=""), token: str = ""):
    """Какие модели лежат в рантайме и какая активна сейчас (+ подсказка
    нужного режима CPU/T4 для каждой)."""
    _auth(x_agent_token, token)
    items = _model_list()
    gpu = os.path.exists("/dev/nvidia0") or bool(shutil.which("nvidia-smi"))
    active = _active_model()
    return {
        "active": active,
        "active_label": _model_label(active) if items and active else "",
        "switching": bool(S.switching),
        "gpu": bool(gpu),
        "models": [{"path": p, "label": _model_label(p),
                    "size_gb": round(os.path.getsize(p) / 2 ** 30, 2),
                    "hint": _model_hint(p, gpu),
                    "active": p == active} for p in items],
    }


class ModelRequest(BaseModel):
    model: str = ""


@app.post("/model")
async def set_model(req: ModelRequest, x_agent_token: str = Header(default=""),
                    token: str = ""):
    """Переключение модели на лету: панель зовёт это напрямую."""
    _auth(x_agent_token, token)
    want = req.model.strip()
    items = _model_list()
    if not items:
        raise HTTPException(404, "в рантайме нет моделей (выполни ячейку A)")
    target = next((p for p in items if p == want), None)
    if target is None:                       # прислали label: "3b", "1.5B"
        low = want.lower()
        target = next((p for p in items if _model_label(p).lower() == low), None)
    if target is None:
        target = next((p for p in items if low and low in os.path.basename(p).lower()), None)
    if target is None:
        raise HTTPException(404, f"модель «{want}» не найдена; есть: "
                                  + ", ".join(_model_label(p) for p in items))
    if target == _active_model() and await _llm_ready(tries=2, gap=1.0):
        return {"ok": True, "active": target, "label": _model_label(target),
                "note": "уже активна"}
    try:
        return await _switch_to(target)
    except Exception as exc:                 # noqa: BLE001
        S.stats["errors"] += 1
        await emit("error", f"переключение не удалось: {exc}")
        raise HTTPException(500, str(exc)[:300])


@app.post("/chat/stream")
async def chat_stream_ep(req: ChatRequest, x_agent_token: str = Header(default=""),
                         token: str = ""):
    """Чат потоком: токены приходят по мере генерации.

    Нужен потому, что бесплатный туннель Cloudflare рвёт молчащий запрос на
    120-й секунде: развёрнутый ответ на 700 токенов на CPU занимает минуты.
    Поток держит соединение живым, поэтому длина ответа больше не ограничена
    временем ожидания первого байта.
    """
    _auth(x_agent_token, token)
    t0 = time.time()
    memory = req.memory if isinstance(req.memory, dict) else {}
    facts = [str(f) for f in (memory.get("facts") or []) if str(f).strip()]
    used = bool(facts or req.profile or req.history)
    await emit("thought", f"Чат (поток): {req.message[:120]}")
    max_tokens = min(int(os.environ.get("THINKING_CHAT_MAX_TOKENS",
                                      str(max(MAX_TOKENS, 700)))),
                      _ctx_limit())      # бюджет не должен превышать окно

    async def gen():
        buf: list[str] = []
        try:
            async with S.sem:
                async for piece in chat_stream(SYSTEM_CHAT,
                                               build_chat_prompt(req), max_tokens):
                    buf.append(piece)
                    yield ("data: " + json.dumps({"type": "token", "text": piece},
                                                 ensure_ascii=False) + "\n\n")
            raw = "".join(buf)
            data = await guard_reply(raw)
            reply = str(data.get("reply") or "").strip()
            yield ("data: " + json.dumps({
                "type": "done",
                "reply": reply,
                "rationale": str(data.get("rationale") or "")[:600],
                "source": "colab", "fallback": False, "memory_used": used,
                "duration_ms": int((time.time() - t0) * 1000),
            }, ensure_ascii=False) + "\n\n")
            await emit("final", f"[чат-поток] {reply[:200]}")
            S.stats["chats"] += 1
        except Exception as exc:             # noqa: BLE001
            S.stats["errors"] += 1
            yield ("data: " + json.dumps({"type": "error", "text": str(exc)[:300]},
                                         ensure_ascii=False) + "\n\n")
    return StreamingResponse(with_heartbeat(gen()), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache",
                                      "X-Accel-Buffering": "no"})


@app.post("/dev")
async def dev(req: DevRequest, x_agent_token: str = Header(default=""),
              token: str = ""):
    """Режим «Разработка» (без потока): модель возвращает предложение
    {action, filename, code, comment}. Файлы НЕ изменяются — применение
    остаётся за человеком на ПК (контроль сохраняется)."""
    _auth(x_agent_token, token)
    t0 = time.time()
    await emit("thought", f"Разработка: {req.message[:120]}")
    max_tokens = min(int(os.environ.get("THINKING_DEV_MAX_TOKENS",
                                      str(max(MAX_TOKENS, 1600)))),
                      _ctx_limit())
    try:
        raw, usage = await chat_json_usage(SYSTEM_DEV, build_dev_prompt(req),
                                           max_tokens)
        data = await dev_data(raw)
    except HTTPException:
        raise
    except Exception as exc:             # noqa: BLE001
        S.stats["errors"] += 1
        await emit("error", f"разработка: {str(exc)[:200]}")
        raise HTTPException(500, str(exc)[:300])
    S.stats["chats"] += 1
    await emit("final", f"[{data['action']}] {data['filename'] or 'без файла'}: "
                        f"{data['comment'][:200]}")
    data["tokens_in"] = int(usage.get("prompt_tokens") or 0)
    data["tokens_out"] = int(usage.get("completion_tokens") or 0)
    data["tokens_estimate"] = not bool(usage)
    data["duration_ms"] = int((time.time() - t0) * 1000)
    data["at"] = utcnow()
    return data


@app.post("/dev/stream")
async def dev_stream(req: DevRequest, x_agent_token: str = Header(default=""),
                     token: str = ""):
    """Тот же запрос, но потоком: JSON приходит токенами, поэтому длинный
    код не обрывается 120-секундным лимитом бесплатного туннеля."""
    _auth(x_agent_token, token)
    t0 = time.time()
    await emit("thought", f"Разработка (поток): {req.message[:120]}")
    max_tokens = int(os.environ.get("THINKING_DEV_MAX_TOKENS",
                                    str(max(MAX_TOKENS, 1600))))

    async def gen():
        buf: list[str] = []
        try:
            async with S.sem:
                async for piece in chat_stream(SYSTEM_DEV, build_dev_prompt(req),
                                               max_tokens):
                    buf.append(piece)
                    yield ("data: " + json.dumps({"type": "token", "text": piece},
                                                 ensure_ascii=False) + "\n\n")
            data = await dev_data("".join(buf))
            yield ("data: " + json.dumps({
                "type": "done", **data,
                "tokens_estimate": True,
                "duration_ms": int((time.time() - t0) * 1000),
                "at": utcnow(),
            }, ensure_ascii=False) + "\n\n")
            await emit("final", f"[{data['action']}] "
                                f"{data['filename'] or 'без файла'}: "
                                f"{data['comment'][:200]}")
            S.stats["chats"] += 1
        except Exception as exc:         # noqa: BLE001
            S.stats["errors"] += 1
            yield ("data: " + json.dumps({"type": "error", "text": str(exc)[:300]},
                                         ensure_ascii=False) + "\n\n")
    return StreamingResponse(with_heartbeat(gen()), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache",
                                      "X-Accel-Buffering": "no"})


@app.get("/dump/{name}")
async def dump(name: str, x_agent_token: str = Header(default=""), token: str = ""):
    """Отдаёт серверный файл из белого списка — чтобы ПК мог снять логи
    в архив без ручного копирования. Требует тот же токен, что и остальное."""
    _auth(x_agent_token, token)
    path = DUMP_FILES.get(name)
    if not path:
        raise HTTPException(404, f"файл «{name}» не в списке: {', '.join(DUMP_FILES)}")
    try:
        with open(path, "rb") as fh:
            data = fh.read()
    except OSError as exc:
        raise HTTPException(404, f"нет файла: {exc}")
    await emit("thought", f"выгружен файл {name} ({len(data)} байт)")
    return StreamingResponse(iter([data]), media_type="application/octet-stream",
                             headers={"Content-Disposition":
                                      f'attachment; filename="{name}"'})


PANEL_HTML = """<!doctype html><html lang="ru"><head><meta charset="utf-8">
<title>Мышление</title><style>
body{margin:0;background:#0d1117;color:#c9d1d9;font:13px/1.45 ui-monospace,Consolas,monospace}
header{display:flex;gap:12px;align-items:center;padding:9px 13px;background:#161b22;border-bottom:1px solid #30363d}
#b{padding:2px 9px;border-radius:11px;background:#3d1418;color:#ff7b72}
#b.on{background:#12351f;color:#7ee787}
main{display:grid;grid-template-columns:1fr 1fr;gap:11px;padding:11px 13px}
section{background:#161b22;border:1px solid #30363d;border-radius:8px;padding:9px;height:84vh;overflow-y:auto}
h2{font-size:11px;margin:0 0 7px;color:#8b949e;letter-spacing:.08em}
.ev{padding:3px 0;border-bottom:1px dashed #21262d;font-size:12px;word-break:break-word}
.t{color:#8b949e;margin-right:6px}
.thought .x{color:#9ecbff}.rationale .x{color:#e6edf3}.plan_step .x{color:#7ee787}
.contradiction .x{color:#ffa657}.final .x{color:#fff;font-weight:600}
.error .x{color:#ff7b72}.token .x{color:#8b949e}
</style></head><body>
<header><b>Субагент «Мышление»</b><span id="b">…</span><span id="m" style="color:#8b949e"></span></header>
<main><section><h2>МЫСЛИ (LIVE)</h2><div id="a"></div></section>
<section><h2>ПОСЛЕДНИЕ СОБЫТИЯ</h2><div id="c"></div></section></main>
<script>
const TOK="__TOKEN__";
function H(){return TOK?{"X-Agent-Token":TOK}:{}}
function add(id,ev){const b=document.getElementById(id);if(b.children.length>300)b.removeChild(b.firstChild);
const d=document.createElement('div');d.className='ev '+(ev.type||'');
const s=document.createElement('span');s.className='t';s.textContent=(ev.ts||'').slice(11,19);
const x=document.createElement('span');x.className='x';x.textContent=ev.text||'';
d.append(s,x);b.appendChild(d);b.scrollTop=b.scrollHeight}
let es=new EventSource("/events/stream?token="+encodeURIComponent(TOK));
es.onopen=()=>{document.getElementById('b').className='on';document.getElementById('b').textContent='ПОДКЛЮЧЕНО'};
es.onerror=()=>{document.getElementById('b').className='';document.getElementById('b').textContent='НЕТ СВЯЗИ'};
es.onmessage=e=>{try{const ev=JSON.parse(e.data);add('c',ev);if(ev.type!=='final')add('a',ev)}catch(x){}};
async function st(){try{const r=await fetch('/health?token='+encodeURIComponent(TOK),{headers:H()});
const h=await r.json();document.getElementById('m').textContent=h.model+' · '+h.gpu+' · uptime '+h.uptime_s+' с'
+' · планов '+h.plans}catch(x){}}
st();setInterval(st,4000);
</script></body></html>"""

In [ ]:
%%writefile /content/tc/cell_d_launch.py
# === Субагент «Мышление», ячейка D (запуск LLM + API + туннель) ===
# Идемпотентна: повторный запуск убивает старые процессы и поднимает заново.
import glob
import json
import os
import re
import secrets
import shutil
import subprocess
import sys
import time

import requests
from google.colab import output


def _token_from_file() -> str:
    """Токен прошлого запуска — чтобы URL/токен не менялись без нужды."""
    try:
        with open(URL_FILE, encoding="utf-8") as fh:
            m = re.search(r"THINKING_TOKEN=(\S+)", fh.read())
            return m.group(1) if m else ""
    except OSError:
        return ""


def _remember(url: str, token: str) -> None:
    try:
        with open(URL_FILE, "w", encoding="utf-8") as fh:
            fh.write(f"THINKING_URL={url}\nTHINKING_TOKEN={token}\n"
                     f"THINKING_PANEL={url}\nupdated={time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    except OSError:
        pass

LOG_LLM = "/content/llm.log"
LOG_API = "/content/thinking_api.log"
URL_FILE = "/content/thinking_url.txt"     # чтобы завтра не искать токен в выводе

# ---- 0) токен и модель -----------------------------------------------------
# Токен можно задать заранее: THINKING_TOKEN = "..." в этой ячейке (или в
# «Переменных» Colab). Тогда ПК не требует нового set-url после перезапуска.
TOKEN = (os.environ.get("THINKING_TOKEN") or _token_from_file() or secrets.token_urlsafe(24))
os.environ["THINKING_TOKEN"] = TOKEN
os.environ.setdefault("THINKING_MODEL", "thinking")

model_file = "/content/thinking_model_path.txt"
models_file = "/content/thinking_models.txt"
env_file = "/content/thinking_env.sh"
ACTIVE_FILE = "/content/thinking_active_model.txt"

# ячейка A могла оставить путь к библиотекам CUDA (если был починен импорт)
if os.path.exists(env_file):
    for line in open(env_file, encoding="utf-8"):
        if line.startswith("export "):
            key, _, val = line[7:].strip().partition("=")
            os.environ[key] = val

assert os.path.exists(model_file), "сначала выполните ячейку A (установка)"
MODEL = open(model_file, encoding="utf-8").read().strip()
assert os.path.exists(MODEL), f"модель не найдена: {MODEL}"
MODELS = [m for m in (open(models_file, encoding="utf-8").read().split("\n")
                      if os.path.exists(models_file) else []) if m] or [MODEL]
print("MODEL:", MODEL, "| запасные:", [os.path.basename(m) for m in MODELS[1:]])

# Длина ответа — единственное, для чего нужны бюджеты: рантайм бесплатный,
# экономить нечего, а короткий ответ ломает контракт (обрезанный JSON не
# разбирается и план превращается в мусор — такое и случилось на 3B с
# бюджетом 260). Поэтому бюджет задаёт ДЛИЗИНУ, а не «боязнь долгого
# ответа»: сервер всё равно прижимает его к n_ctx (THINKING_CTX).
# Проверка идёт СВЕРХУ ВНИЗ, от самого длинного признака к короткому: иначе
# «a3b» внутри Qwen3-30B-A3B поймал бы трёхбайтовый бюджет, а «7b» внутри
# «27b» — семёрку.
MODEL_BUDGETS = [
    # (признак в имени, ответ, поток чата, поток разработки)
    ("30b-a3b", "1200", "2400", "3000"),
    ("32b", "1200", "2400", "3000"),
    ("27b", "1200", "2400", "3000"),
    ("14b", "1000", "2000", "2600"),
    ("7b", "900", "1800", "2400"),
    ("3b", "700", "1400", "2000"),
    ("1.5b", "500", "1000", "1600"),
]
_name = os.path.basename(MODEL).lower()
_match = next((b for b in MODEL_BUDGETS if b[0] in _name),
              ("3b", "700", "1400", "2000"))      # запасной вариант
os.environ.setdefault("THINKING_MAX_TOKENS", _match[1])
# Для потокового чата (/chat/stream) можно больше: токены идут по мере генерации,
# туннель Cloudflare не рвёт соединение, а человек видит текст сразу.
os.environ.setdefault("THINKING_CHAT_MAX_TOKENS", _match[2])
# Режим разработчика возвращает код целиком — тут обрезать нельзя.
os.environ.setdefault("THINKING_DEV_MAX_TOKENS", _match[3])
# ВНИМАНИЕ: это предварительный расчёт по ПЕРВОЙ кандидатуре. Если она не
# поднимется и ячейка возьмёт запасную — бюджет пересчитается ниже, уже по
# той модели, что реально работает.
# Активная модель — в файле: его читает и сервер (вкладка «Модели» в панели),
# и сторож, чтобы переключение не слетело после перезапуска движка.
with open(ACTIVE_FILE, "w", encoding="utf-8") as fh:
    fh.write(MODEL)
print("Бюджет по первой кандидатуре:", os.environ["THINKING_MAX_TOKENS"],
      "/", os.environ["THINKING_CHAT_MAX_TOKENS"], "/",
      os.environ["THINKING_DEV_MAX_TOKENS"],
      "(признак:", _match[0] + ") — уточним после выбора модели")

# ---- 1) остановка прошлых экземпляров -------------------------------------
subprocess.run(["pkill", "-f", "llama_cpp.server"], capture_output=True)
subprocess.run(["pkill", "-f", "uvicorn thinking_server"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(2)


def wait_http(url, tries=150, gap=2, payload=None):
    for _ in range(tries):
        try:
            r = requests.get(url, timeout=4) if payload is None else None
            if r is not None and r.status_code == 200:
                return r
        except Exception:
            pass
        time.sleep(gap)
    return None


def _avail_gb() -> float:
    """Сколько памяти реально свободно (ГБ) — по факту, а не по памятке.

    Colab-железо меняется, и модель, которая влезла вчера, может не влезть
    сегодня. Смотрим MemAvailable, потому что после загрузки модели сами
    страницы весов и есть «занятая» память.
    """
    try:
        for line in open("/proc/meminfo", encoding="utf-8"):
            if line.startswith("MemAvailable:"):
                return int(line.split()[1]) / 1024 ** 2
    except Exception:
        pass
    return 0.0


# ---- 2) LLM backend на :8001 ----------------------------------------------
# Контекст 4096 на CPU (быстрее считать) и 8192 на GPU — качество не страдает.
GPU = os.path.exists("/dev/nvidia0") or bool(shutil.which("nvidia-smi"))
CTX = "8192" if GPU else "4096"
# Пишем контекст в env-файл: его читает _llm_cmd при сторожевом перезапуске
# (ячейка C), чтобы n_ctx не разъехался между запуском и перезапуском (AUD-08)
with open(env_file, "a", encoding="utf-8") as fh:
    fh.write(f'export THINKING_CTX="{CTX}"\n')
llm_cmd = [sys.executable, "-m", "llama_cpp.server",
           "--model", MODEL, "--n_ctx", CTX, "--n_gpu_layers", "-1",
           "--host", "127.0.0.1", "--port", "8001", "--model_alias", "thinking",
           "--n_threads", "4", "--n_batch", "512", "--verbose", "False"]
open(LOG_LLM, "a", encoding="utf-8").write(f"\n--- {time.strftime('%H:%M:%S')} {llm_cmd}\n")

# Модель могла не подойти — тогда берём следующую из списка (их до трёх).
llm_proc = None
health = None
for attempt, cand in enumerate(MODELS, 1):
    size_gb = os.path.getsize(cand) / 1024 ** 3
    cmd = list(llm_cmd)
    cmd[cmd.index("--model") + 1] = cand
    llm_proc = subprocess.Popen(cmd, stdout=open(LOG_LLM, "a", encoding="utf-8"),
                                stderr=subprocess.STDOUT)
    print(f"llm pid {llm_proc.pid} ({attempt}/{len(MODELS)}: "
          f"{os.path.basename(cand)}, {size_gb:.1f} ГБ)")
    # Большая модель читается с диска долго: 17 ГБ успевает не всегда за 90 с,
    # и раньше мы отвергали её как «не поднялась», не дав дописать веса.
    # Время ожидания растёт от размера файла.
    tries = 45 + int(size_gb * 12)
    # Показываем требуемую память ДО ожидания. Примерная оценка: веса плюс
    # KV-кэш (около 10% весов при n_ctx 4096). Иначе приходилось ждать
    # несколько минут, чтобы потом увидеть отказ.
    kv_gb = size_gb * 0.10 * (int(CTX) / 4096)
    need_gb = size_gb + kv_gb
    print(f"  ждём подъёма до {tries * 2 // 60} мин "
          f"(размер {size_gb:.1f} ГБ, веса+KV ≈ {need_gb:.1f} ГБ при RAM ~12 ГБ)")
    # Память проверяем по факту, а не по памятке: Colab даёт разное железо.
    # Модель, которая не помещается, не «не поднимется» — она поднимется и
    # будет работать в подкачке страниц, то есть в десятки раз медленнее.
    # Прогон 04.10: 14B (8.4 ГБ) при свободных 3.0 ГБ дала 0.05 ток/с,
    # то есть 20 раз медленнее 3B. Молча мириться с этим нельзя.
    if _avail_gb() and need_gb > _avail_gb() * 1.1:
        print(f"  (!) ВНИМАНИЕ: весам нужно ≈{need_gb:.1f} ГБ, а свободно только "
              f"{_avail_gb():.1f} ГБ. Будет подкачка страниц с диска — скорость "
              "упадёт в десятки раз (замерено 0.05 ток/с на 14B).")
        print("      Профиль gpu (7B, 4.7 ГБ) или light (3B, 1.8 ГБ) влезут "
              "свободно. Модель всё равно попробуем — но ответ будет долгим.")
    health = wait_http("http://127.0.0.1:8001/v1/models", tries=tries, gap=2)
    if health is not None:
        MODEL = cand
        break
    # Не молчим: показываем ПОЧЕМУ модель не поднялась. Раньше здесь
    # печаталось только «не поднялась», и причину приходилось угадывать.
    _why = open(LOG_LLM, encoding="utf-8", errors="replace").readlines()[-25:]
    print(f"  {os.path.basename(cand)} не поднялась. Последние строки лога:")
    for _line in _why:
        print("   |", _line.rstrip())
    # Код возврата — единственный источник правды, когда лог ПУСТ. Именно так
    # выглядит прогон 04.10: лог содержал только строку команды, ни слова об
    # ошибке, поэтому пришлось выбирать между «не хватило памяти» и «движок не
    # знает архитектуру». poll() < 0 значит «убит сигналом»:
    #   -9  OOM-киллер: ядро не дало памяти — крупная модель в RAM не влезет;
    #   -11 segfault: чаще всего старая сборка llama.cpp не знает Qwen3.
    # poll() > 0 — процесс вышел сам, None — жив, то есть просто не успел.
    rc = llm_proc.poll()
    if rc is None:
        print("   | процесс ещё ЖИВ: не упал, а не успел поднять порт за отведённое время")
    elif rc < 0:
        import signal as _sig
        try:
            _name_sig = _sig.Signals(-rc).name
        except Exception:
            _name_sig = f"сигнал {-rc}"
        print(f"   | ПРОЦЕСС УБИТ: {_name_sig} (код {rc})")
        if rc == -9:
            print("   | это OOM-киллер: памяти не хватило. Модель крупнее RAM "
                  "(~12 ГБ) так не запустится — бери профиль strong (14B, 8 ГБ)")
        elif rc == -11:
            print("   | это segfault: скорее всего движок не знает архитектуру "
                  "модели (Qwen3 требует свежей сборки llama.cpp). Профиль "
                  "strong (Qwen2.5-14B) поддерживается любой сборкой")
    else:
        print(f"   | процесс завершился сам, код возврата {rc}")
    if size_gb > 12:
        print(f"  (!) модель {size_gb:.1f} ГБ при RAM ~12 ГБ: может не хватить "
              "памяти. Возьми профиль strong (14B, 8 ГБ) или gpu.")
    try:
        llm_proc.kill()
    except Exception:
        pass
    time.sleep(3)          # порт 8001 должен освободиться до следующей попытки

if health is None:
    print("LLM НЕ ПОДНЯЛСЯ. Последние 60 строк лога:")
    print("\n".join(open(LOG_LLM, encoding="utf-8", errors="replace").readlines()[-60:]))
    raise RuntimeError("llama_cpp.server не стартовал")
print("MODEL в работе:", os.path.basename(MODEL))
# Бюджет пересчитываем ПОСЛЕ выбора модели: раньше он считался по первой
# кандидатуре, и если поднималась запасная — работала с чужими лимитами
# (прогон 04.10: поднялась 3B, а бюджеты остались от 30B — 1200/2400/3000).
_name = os.path.basename(MODEL).lower()
_match = next((b for b in MODEL_BUDGETS if b[0] in _name),
              ("3b", "700", "1400", "2000"))
os.environ["THINKING_MAX_TOKENS"] = _match[1]
os.environ["THINKING_CHAT_MAX_TOKENS"] = _match[2]
os.environ["THINKING_DEV_MAX_TOKENS"] = _match[3]
print("ЛИМИТ ОТВЕТА (по модели в работе):", os.environ["THINKING_MAX_TOKENS"],
      "/", os.environ["THINKING_CHAT_MAX_TOKENS"], "/",
      os.environ["THINKING_DEV_MAX_TOKENS"], "признак:", _match[0])
# сторож в конце ячейки перезапускает llm_cmd — значит в нём должна быть
# та модель, которая реально поднялась, а не первая из списка
llm_cmd[llm_cmd.index("--model") + 1] = MODEL
# хвосты процессов для сторожа: он отличает «процесс жив» от «процесса нет»
PROCS: dict = {"llm": llm_proc}
print("LLM ready:", health.text[:200])

# ---- 3) смоук-тест генерации ----------------------------------------------
# На CPU модель может отвечать дольше 180 с или упасть на старте — это не
# повод ронять ячейку целиком: API и туннель поднимаются дальше, а проверить
# генерацию можно руками (AUD-24).
t0 = time.time()
try:
    smoke = requests.post(
        "http://127.0.0.1:8001/v1/chat/completions",
        json={"model": "thinking", "temperature": 0,
              "max_tokens": 24, "messages": [{"role": "user",
                                              "content": "Ответь ровно одним словом: готов"}]},
        timeout=180)
    dt = time.time() - t0
    txt = smoke.json()["choices"][0]["message"]["content"]
    print(f"смоук: {dt:.1f} с -> {txt[:80]!r}")
except Exception as _smoke_exc:
    print(f"! смоук не прошёл: {_smoke_exc}")
    print("  модель отвечает медленно или ещё грузится — проверь руками:")
    print("  curl -s http://127.0.0.1:8001/v1/chat/completions -X POST "
          "-H 'Content-Type: application/json' "
          "-d '{\"model\":\"thinking\",\"max_tokens\":24,"
          "\"messages\":[{\"role\":\"user\",\"content\":\"привет\"}]}'")

# ---- 4) API на :8000 -------------------------------------------------------
assert os.path.exists("/content/thinking_server.py"), "сначала ячейка C (сервер)"
os.environ["THINKING_UPSTREAM"] = "http://127.0.0.1:8001/v1/chat/completions"
api_cmd = [sys.executable, "-m", "uvicorn", "thinking_server:app",
           "--host", "127.0.0.1", "--port", "8000", "--app-dir", "/content"]
open(LOG_API, "a", encoding="utf-8").write(f"\n--- {time.strftime('%H:%M:%S')} {api_cmd}\n")
api_proc = subprocess.Popen(api_cmd, cwd="/content",
                            stdout=open(LOG_API, "a", encoding="utf-8"),
                            stderr=subprocess.STDOUT)
print("api pid", api_proc.pid)

hdr = {"X-Agent-Token": TOKEN}
health_api = wait_http("http://127.0.0.1:8000/health", tries=60, gap=1)
# /health требует токен, поэтому просто бьём до тех пор, пока не получим 200
for _ in range(60):
    try:
        r = requests.get("http://127.0.0.1:8000/health", headers=hdr, timeout=4)
        if r.status_code == 200:
            health_api = r
            break
    except Exception:
        pass
    time.sleep(1)
if health_api is None:
    print("\n".join(open(LOG_API, encoding="utf-8", errors="replace").readlines()[-40:]))
    raise RuntimeError("uvicorn thinking_server не стартовал")
print("API ready:", health_api.text[:300])

# ---- 5) публичные URL ------------------------------------------------------
try:
    panel_url = output.serve_kernel_port_as_window(8000)
except Exception as exc:
    panel_url = None
    print("proxy window:", exc)

tunnel = None
tunnel_kind = "нет"
try:
    if not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run(["wget", "-q",
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                        "-O", "/usr/local/bin/cloudflared"], check=True)
        os.chmod("/usr/local/bin/cloudflared", 0o755)
    open("/content/tunnel.log", "w", encoding="utf-8").close()

    # 1) ИМЕНОВАННЫЙ туннель Cloudflare — постоянный адрес на сутки и дольше.
    #    Задайте переменную CLOUDFLARE_TUNNEL_TOKEN (dashboard → tunnels → token)
    #    или вставьте токен ниже: CF_TOKEN = "eyJhIjoi..."
    CF_TOKEN = os.environ.get("CLOUDFLARE_TUNNEL_TOKEN", "")
    if CF_TOKEN:
        subprocess.Popen(["nohup", "cloudflared", "tunnel", "--no-autoupdate", "run",
                          "--token", CF_TOKEN],
                         stdout=open("/content/tunnel.log", "w", encoding="utf-8"),
                         stderr=subprocess.STDOUT, start_new_session=True)
        time.sleep(12)
        txt = open("/content/tunnel.log", encoding="utf-8", errors="replace").read()
        m = re.search(r"https://[a-z0-9.-]+\.(?:trycloudflare\.com|"
                      r"[a-z0-9-]+\.[a-z0-9-]+\.workers\.dev|[a-z0-9.-]+\.cfargotunnel\.com)", txt)
        if m:
            tunnel, tunnel_kind = m.group(0), "именованный (постоянный)"
        else:
            print("ИМЕНОВАННЫЙ туннель: адрес не распознан в логе, смотрите /content/tunnel.log")
    # 2) Быстрый туннель — бесплатно и без аккаунта, но адрес случайный на каждый запуск
    if not tunnel:
        # --protocol http2: quick-туннель по умолчанию ходит по QUIC/UDP, а на
        # Colab ядра дают крошечный UDP-буфер (quic-go пишет «wanted 7168 kiB,
        # got 416 kiB») — SSE-потоки посреди передачи замирали, done не доходил.
        # HTTP/2 идёт по TCP и этих потерь не видит.
        subprocess.Popen(["nohup", "cloudflared", "tunnel", "--no-autoupdate",
                          "--protocol", "http2",
                          "--url", "http://localhost:8000"],
                         stdout=open("/content/tunnel.log", "a", encoding="utf-8"),
                         stderr=subprocess.STDOUT, start_new_session=True)
        for _ in range(40):
            txt = open("/content/tunnel.log", encoding="utf-8", errors="replace").read()
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
            if m:
                tunnel, tunnel_kind = m.group(0), "быстрый (случайный адрес)"
                break
            time.sleep(2)
except Exception as exc:
    print("tunnel error:", exc)

url = tunnel or (str(panel_url) if panel_url else "")
_remember(url, TOKEN)

# ---- 6) проверка через публичный URL --------------------------------------
if url:
    for _ in range(20):
        try:
            r = requests.get(url.rstrip("/") + "/health",
                             headers={"X-Agent-Token": TOKEN}, timeout=8)
            if r.status_code == 200:
                print("публичный health:", r.text[:200])
                break
        except Exception:
            pass
        time.sleep(3)

print("=" * 72)
print("THINKING_URL=", url)
print("THINKING_TOKEN=", TOKEN)
print("THINKING_PANEL=", url)
print("ТУННЕЛЬ:", tunnel_kind)
print("=" * 72)

# ---- 7) сторож: если LLM/API умрёт — поднимем сами (ноутбук не ждёт вас) ----
import threading                                          # noqa: E402


def _alive(url_: str) -> bool:
    try:
        return requests.get(url_.rstrip("/") + "/health",
                            headers={"X-Agent-Token": TOKEN}, timeout=6).status_code == 200
    except Exception:
        return False


def _watch() -> None:
    api_fail = llm_fail = 0
    while True:
        time.sleep(60)
        try:
            if _alive("http://127.0.0.1:8000"):
                api_fail = llm_fail = 0
                continue
            api_fail += 1
            print(time.strftime("%H:%M:%S"), "API не отвечает,", api_fail, "/3")
            if api_fail >= 3:
                # Жив ли движок — вопрос не к HTTP, а к ПРОЦЕССУ. Запрос к
                # /v1/models с таймаутом 6 с врёт: занятая генерацией модель
                # (или модель, которая подгружает страницы весов с диска) за
                # 6 с не ответит, и сторож решил бы, что движок умер, и
                # перезапустил его. На медленной модели ответ так и не доехал
                # бы никогда — сторож убивал бы его каждые пару минут.
                # poll() отвечает мгновенно и однозначно.
                proc = PROCS.get("llm")
                if proc is not None and proc.poll() is None:
                    llm_fail = 0
                    print(time.strftime("%H:%M:%S"),
                          "LLM занят, но процесс жив — не трогаю")
                else:
                    try:
                        requests.get("http://127.0.0.1:8001/v1/models", timeout=20)
                        llm_fail = 0
                    except Exception:
                        llm_fail += 1
                if llm_fail >= 2:
                    print(time.strftime("%H:%M:%S"), "поднимаю LLM заново")
                    # ту модель, которую выбрали в панели, а не первоначальную
                    active = MODEL
                    try:
                        cur = open(ACTIVE_FILE, encoding="utf-8").read().strip()
                        if cur and os.path.exists(cur):
                            active = cur
                    except OSError:
                        pass
                    cmd = list(llm_cmd)
                    cmd[cmd.index("--model") + 1] = active
                    PROCS["llm"] = subprocess.Popen(
                        cmd, stdout=open(LOG_LLM, "a", encoding="utf-8"),
                        stderr=subprocess.STDOUT)
                    time.sleep(90)
                subprocess.Popen(api_cmd, cwd="/content",
                                 stdout=open(LOG_API, "a", encoding="utf-8"),
                                 stderr=subprocess.STDOUT)
                print(time.strftime("%H:%M:%S"), "API перезапущен")
                api_fail = llm_fail = 0
        except Exception as exc:
            print("watchdog:", type(exc).__name__)


threading.Thread(target=_watch, daemon=True).start()
print("сторож запущен: раз в 60 с проверяет /health и перезапускает упавшее")

print("\nЗАВТРА (когда откроешь этот ноутбук заново):")
print("  1) Runtime → Change runtime type → T4 GPU")
print("  2) выполни ячейки A, C, D подряд (модель обычно уже лежит в Drive —")
print("     тогда загрузка пропускается, весь запуск занимает пару минут)")
print("  3) скопируй THINKING_URL и THINKING_TOKEN из рамки выше")
print("  4) на ПК:  python tools/thinking_cli.py set-url <URL> <TOKEN>")
print("  5) проверка: python tools/thinking_cli.py doctor")
print("  Если задал CLOUDFLARE_TUNNEL_TOKEN — адрес будет тот же, set-url не нужен.")

In [ ]:
%%writefile /content/tc/cell_e_background.py
# === Субагент «Мышление», ячейка E (фон) ===
# Снапшот событий в файл + keep-alive. Всё в фоновых потоках:
# ячейка НЕ блокирует ядро, остальные ячейки остаются работоспособными.
import json
import os
import threading
import time
import urllib.request
from datetime import datetime

SNAP = "/content/thinking_snapshot.json"
TOKEN = os.environ.get("THINKING_TOKEN", "")


def _snapshot_loop():
    """Раз в минуту думпает события — переживает рестарт uvicorn."""
    while True:
        try:
            req = urllib.request.Request(
                "http://127.0.0.1:8000/events?tail=300",
                headers={"X-Agent-Token": TOKEN})
            data = json.load(urllib.request.urlopen(req, timeout=10))
            tmp = SNAP + ".tmp"
            with open(tmp, "w", encoding="utf-8") as fh:
                json.dump(data, fh, ensure_ascii=False)
            os.replace(tmp, SNAP)
            print(datetime.now().strftime("%H:%M:%S"), "snapshot:",
                  len(data.get("events", [])), flush=True)
        except Exception as exc:
            print("snapshot error:", type(exc).__name__, flush=True)
        time.sleep(60)


def _keepalive():
    for _ in range(144):          # 12 часов, печатает раз в 5 минут
        time.sleep(300)
        print(datetime.now().strftime("%H:%M:%S"), "keep-alive", flush=True)


if TOKEN:
    threading.Thread(target=_snapshot_loop, daemon=True).start()
    print("фон запущен: снапшот событий каждые 60 с ->", SNAP)
else:
    print("токен не задан (сначала ячейка запуска) — снапшот не нужен")
threading.Thread(target=_keepalive, daemon=True).start()
print("keep-alive: 12 ч в фоне (ядро не блокируется)")

# --- опционально: URL/токен на Google Drive, чтобы новая сессия Colab
#     подхватила их сама (примонтируйте Drive один раз в ячейке выше) -------
try:
    drive_dir = "/content/drive/MyDrive"
    if os.path.isdir(drive_dir):
        import shutil
        shutil.copy("/content/thinking_url.txt", os.path.join(drive_dir, "thinking_url.txt"))
        print("URL/токен сохранены на Google Drive -> thinking_url.txt")
    else:
        print("Drive не примонтирован — URL/токен остались только в /content")
except Exception as exc:
    print("Drive не сохранил:", type(exc).__name__, flush=True)

# --- подсказка: восстановить адрес/токен в новой сессии --------------------
print("\nЕсли завтра откроешь ноутбук заново: выполни ячейки A, C, D.")
print("Перед ячейкой D можно задать токен, чтобы адрес ПК не менялся:")
print('    THINKING_TOKEN = "<токен из thinking_url.txt>"')
print("Файл с URL и токеном: /content/thinking_url.txt (и Google Drive, если смонтирован)")

In [ ]:
%%writefile /content/tc/cell_f_stop.py
# === Субагент «Мышление» — остановка ===
# Останавливает только субагента (LLM, API, туннель). Ячейки генерации волн
# в других ноутбуках не затронуты.
#
# ВАЖНО: эта ячейка входит в «Run all», а раньше из-за неё «Run all» поднимал
# субагента и тут же его гасил (план возвращался, но сервисы уже мертвы).
# Поэтому остановка требует явного разрешения: выполните ячейку, когда
#   THINKING_STOP = "1"
# задана в «Переменных» Colab (или в той же ячейке выше), — тогда она сработает.
import os
import subprocess


def stop() -> None:
    """Останавливает сервисы субагента, если остановка разрешена.

    Раньше здесь был выход через SystemExit(0), и это давало две неприятные
    вещи при каждом «Run all»: IPython печатал «An exception has occurred,
    use %tb» и предупреждение «To exit; use 'exit', 'quit', or Ctrl-D».
    SystemExit внутри ячейки IPython принимает за просьбу выключить яДРО — то
    есть отказ от остановки выглядел как авария. Отказ — обычный исход,
    поэтому выходим простым return.
    """
    if os.environ.get("THINKING_STOP", "").strip() not in ("1", "yes", "true"):
        print("субагент НЕ остановлен: остановка при Run all отключена.\n"
              "  Чтобы остановить его по-настоящему, выполни эту ячейку при\n"
              '  THINKING_STOP = "1" (Переменные Colab) — либо в терминале:\n'
              "  pkill -f thinking_server; pkill -f llama_cpp.server; pkill -f cloudflared")
        return
    for pat in ("thinking_server", "llama_cpp.server", "uvicorn thinking_server",
                "cloudflared"):
        subprocess.run(["pkill", "-f", pat], capture_output=True)
    print("субагент остановлен (ячейки генерации волн не затронуты)")


stop()

In [ ]:
#@title 2/7 · Ячейка A — окружение, сборка, модель
import os, runpy
# ── Что скачивать (поменяй строку и перезапуни) ──
# light      — 3B + 1.5B (~2 ГБ), быстро, для CPU
# gpu        — 7B + 3B + 1.5B (~6 ГБ), для T4
# big        — Qwen3-30B-A3B (17 ГБ) + 3B: MoE, активны 3B,
#               отвечает как большая, считает почти как 3B
# strong     — Qwen2.5-14B + Coder-14B (17 ГБ), влезают в RAM
# coder      — Qwen2.5-Coder-32B (18 ГБ), лучший для кода
# uncensored — сборки без цензуры + обычная 3B
# all        — всё сразу (~60 ГБ, НЕ советуем)
PROFILE = "big"
os.environ["THINKING_PROFILE"] = PROFILE
runpy.run_path('/content/tc/cell_a_setup.py', run_name='__main__')

In [ ]:
#@title 3/7 · Ячейка C — сборка API-сервера
import pathlib
src = pathlib.Path('/content/thinking_server.py')
print('записан /content/thinking_server.py:', len(src.read_text(encoding='utf-8').splitlines()), 'строк')

In [ ]:
#@title 4/7 · Ячейка D — LLM + API + туннель + сторож
import os, runpy
os.environ.setdefault('THINKING_TOKEN', '')   # впишите токен, чтобы адрес не менялся
runpy.run_path('/content/tc/cell_d_launch.py', run_name='__main__')

In [ ]:
#@title 5/7 · Ячейка E — фон (снапшот + keep-alive)
import runpy; runpy.run_path('/content/tc/cell_e_background.py', run_name='__main__')

In [ ]:
#@title 6/7 · Ячейка F — остановка (по необходимости)
import runpy; runpy.run_path('/content/tc/cell_f_stop.py', run_name='__main__')

In [ ]:
#@title 7/7 · Как открыть веб-панель (адрес туннеля + команда для ПК)
import pathlib, re
f = pathlib.Path("/content/thinking_url.txt")
data = f.read_text(encoding="utf-8") if f.exists() else ""
url = (re.search(r"THINKING_URL=(\S+)", data) or [None, ""])[1]
token = (re.search(r"THINKING_TOKEN=(\S+)", data) or [None, ""])[1]
print("=" * 66)
print("  АДРЕС ТУННЕЛЯ:", url or "не найден — выполни ячейку 4/7 (D)")
print("=" * 66)
print()
if url:
    print("ШАГ 1. На ПК скопируй и выполни одну команду:")
    print()
    print(f'    python tools/thinking_cli.py set-url "{url}" "{token}"')
    print()
    print("ШАГ 2. Открой панель (в этом же окне PowerShell):")
    print()
    print("    python tools/thinking_cli.py panel")
    print()
    print("    -> откроется http://127.0.0.1:8765  (панель на 9 вкладок)")
    print()
    print("ШАГ 3. Проверь, что субагент отвечает:")
    print()
    print("    python tools/thinking_cli.py doctor")
    print()
    print("ШАГ 4. Если субагентом пользуется ещё и код-агент — просто вставь")
    print("       ему строку «АДРЕС ТУННЕЛЯ» и токен выше из этого вывода:")
    print("       он выполнит set-url и прогонит полный тест (план, рефлексия,")
    print("       отчёт, память). Отдельно ничего настраивать не нужно.")
else:
    print("Адреса пока нет. Выполни ячейку 4/7 (D) — она поднимает туннель,")
    print("потом вернись сюда и выполни эту ячейку ещё раз.")


---

## Где взять адрес (чтобы не искать его каждый раз)

Выполните **ячейку 7/7** в самом конце ноутбука — она напечатает всё готовое
копированием:

```
АДРЕС ТУННЕЛЯ: https://xxxx-xxxx.trycloudflare.com
```

**Веб-панель на ПК** (самая частая задача):

```bash
python tools/thinking_cli.py panel
```

→ откроется **http://127.0.0.1:8765** — панель на 9 вкладок (Отчёты, История,
Диалог, Память, Токены, Выгода, Модели, Разработка, Лента). Токен подставится
сам из `config/thinking.local.json`.

**Связь не работает / адрес не подходит:**

```bash
python tools/thinking_cli.py set-url <АДРЕС> <ТОКЕН>   # адрес и токен — из ячейки 7/7
python tools/thinking_cli.py doctor                     # что сломано и где смотреть
python tools/thinking_cli.py panel                     # открыть панель
```

`doctor` печатает причину по-человечески: не отвечает Colab, отвечает, но
молчит, или токен не тот.

**Панель молчит при открытии** — это нормально: интерфейс и история лежат
на ПК. Colab нужен только чтобы отвечать на новые запросы.
